In [ ]:
# @title Cell 1: Initial Setup and API Credentials

# --- Installations ---
!pip install requests pyyaml python-dotenv -q
print("Libraries installed.")

# --- Core Imports ---
import requests
import json
import os
import getpass
import time # Keep time here as it's used in polling loops later
from dotenv import load_dotenv
from google.colab import userdata


# --- Load Environment Variables (Optional) ---
# load_dotenv() # Uncomment if using a .env file locally

# --- DigitalOcean API Token ---
# Use getpass to securely input your token when running the cell
try:
    # Try getting the token from Colab secrets first
    DIGITALOCEAN_TOKEN = userdata.get('DO_API_TOKEN')
    if not DIGITALOCEAN_TOKEN:
        print("Could not find 'DO_API_TOKEN' in Colab secrets.")
        print("Please provide your DigitalOcean API Token.")
        DIGITALOCEAN_TOKEN = getpass.getpass("Enter your DigitalOcean API Token: ")
# If userdata.get fails (e.g., not in Colab), fall back to getpass
except NameError: # userdata might not be defined if not in Colab
     print("Not running in Google Colab or 'userdata' unavailable.")
     print("Please provide your DigitalOcean API Token.")
     DIGITALOCEAN_TOKEN = getpass.getpass("Enter your DigitalOcean API Token: ")
except Exception as e:
    print(f"An error occurred trying to get the token: {e}")
    print("Please provide your DigitalOcean API Token.")
    DIGITALOCEAN_TOKEN = getpass.getpass("Enter your DigitalOcean API Token: ")


if not DIGITALOCEAN_TOKEN:
    raise ValueError("DigitalOcean API Token is required to proceed.")
else:
    print("DigitalOcean API Token received.")
    # Store securely for other cells (Comment moved to the line above)
    %store DIGITALOCEAN_TOKEN

# --- API Base Setup ---
DO_API_BASE_URL = "https://api.digitalocean.com/v2"
HEADERS = {
    "Authorization": f"Bearer {DIGITALOCEAN_TOKEN}",
    "Content-Type": "application/json"
}
# Store for other cells (Comment moved to the line above)
%store DO_API_BASE_URL HEADERS

# --- Helper Function ---
def check_do_response(response, step_name):
    """Checks DigitalOcean API response status and prints messages."""
    print(f"--- Status Check: {step_name} ---")
    try:
        response.raise_for_status()  # Raises HTTPError for bad responses (4xx or 5xx)
        print(f"SUCCESS: Request successful (Status Code: {response.status_code}).")
        try:
            return response.json()
        except json.JSONDecodeError:
            print("WARNING: Response was successful but contained no JSON data.")
            return None
    except requests.exceptions.HTTPError as err:
        print(f"ERROR: Request failed.")
        print(f"Status Code: {response.status_code}")
        print(f"Reason: {response.reason}")
        try:
            error_details = response.json()
            print(f"Error Details: {json.dumps(error_details, indent=2)}")
        except json.JSONDecodeError:
            print(f"Response Content: {response.text}")
        # It's often better to let the calling code decide whether to stop,
        # so consider removing 'raise err' if you want the script to potentially continue
        # raise err # Re-raise the exception to stop the script
        return None # Return None on HTTP error to allow checking
    except Exception as e:
        print(f"An unexpected error occurred during response check: {e}")
        # Consider removing 'raise e' as well for similar reasons as above
        # raise e
        return None # Return None on other errors

print("Initial setup complete. API Token and Headers stored.")

In [ ]:
# @title Cell 2: Clone Private Repository using PAT

# --- Imports ---
import os
import getpass
from urllib.parse import urlparse, urlunparse

# --- Configuration ---
# Provide the standard HTTPS URL of your PRIVATE repository
GIT_REPO_URL = "https://github.com/your-org/podcast-generator"
REPO_NAME = GIT_REPO_URL.split('/')[-1].replace('.git', '')

# --- Logic ---
print("--- Cloning Private GitHub Repository ---")
print("You will need a GitHub Personal Access Token (PAT) with the 'repo' scope.")
print("Create one here: https://github.com/settings/tokens?type=beta (Fine-grained) or https://github.com/settings/tokens (Classic)")

# Securely get the PAT from the user
github_pat = userdata.get('GITHUB_PAT')

if not github_pat:
    print("ERROR: GitHub PAT is required to clone a private repository.")
else:
    try:
        # Construct the authenticated URL
        # Parses the original URL and inserts the token before the host
        parsed_url = urlparse(GIT_REPO_URL)
        # netloc format becomes '<token>@hostname'
        authenticated_netloc = f"{github_pat}@{parsed_url.netloc}"
        authenticated_url_parts = parsed_url._replace(netloc=authenticated_netloc)
        authenticated_url = urlunparse(authenticated_url_parts)

        # Clean up previous clone if it exists
        if os.path.exists(REPO_NAME):
          print(f"Removing existing directory: {REPO_NAME}")
          !rm -rf {REPO_NAME}

        # Clone using the authenticated URL (use -q for less verbose output)
        print(f"\nCloning repository: {GIT_REPO_URL} (using PAT authentication)")
        # Avoid printing the authenticated_url to keep the token out of output
        clone_command = f"git clone -q {authenticated_url}"

        # Execute the clone command
        # Use os.system to get exit code
        exit_code = os.system(clone_command)

        # Check if clone was successful
        if exit_code == 0 and os.path.exists(REPO_NAME):
          print(f"\nSuccessfully cloned private repository into directory: {REPO_NAME}")
          print("Files (first level):")
          !ls {REPO_NAME}
          #Store repo name if needed later
          %store REPO_NAME
        else:
          print(f"\nERROR: Failed to clone repository. Exit code: {exit_code}")
          print("Possible reasons:")
          print("- Incorrect PAT or insufficient permissions (needs 'repo' scope).")
          print("- Incorrect repository URL.")
          print("- Network issues.")
          # Optionally remove partially created directory on failure
          if os.path.exists(REPO_NAME):
              print(f"Removing potentially incomplete directory: {REPO_NAME}")
              !rm -rf {REPO_NAME}

    except Exception as e:
        print(f"\n An unexpected error occurred during the cloning process: {e}")
        # Clean up directory if it exists after an error
        if os.path.exists(REPO_NAME):
           print(f"Removing potentially incomplete directory: {REPO_NAME}")
           !rm -rf {REPO_NAME}

In [ ]:
# @title Cell 3: Step 1 - Create DigitalOcean Kubernetes Cluster

# --- Imports ---
import requests
import json
import time
import sys # Import sys for sys.exit on critical errors

# --- Retrieve stored variables needed here ---
# We only need the base URL and headers from Cell 1
%store -r DO_API_BASE_URL HEADERS

# --- Helper Function (Copied from Cell 1) ---
# Include the function definition directly in this cell
def check_do_response(response, step_name):
    """Checks DigitalOcean API response status and prints messages."""
    print(f"--- Status Check: {step_name} ---")
    try:
        response.raise_for_status()  # Raises HTTPError for bad responses (4xx or 5xx)
        print(f"SUCCESS: Request successful (Status Code: {response.status_code}).")
        try:
            # Handle cases where success response might be empty (e.g., 204 No Content)
            if response.status_code != 204:
                 return response.json()
            else:
                 print("INFO: Request successful with empty response body (Status 204).")
                 return None
        except json.JSONDecodeError:
            print("WARNING: Response was successful but contained non-JSON data or was empty.")
            return None
    except requests.exceptions.HTTPError as err:
        print(f"ERROR: Request failed.")
        print(f"Status Code: {response.status_code}")
        print(f"Reason: {response.reason}")
        try:
            error_details = response.json()
            print(f"Error Details: {json.dumps(error_details, indent=2)}")
        except json.JSONDecodeError:
            print(f"Response Content: {response.text}")
        # Re-raise the specific HTTPError to be caught by the main try/except
        raise err
    except Exception as e:
        print(f"An unexpected error occurred during response check: {type(e).__name__} - {e}")
        # Re-raise the exception
        raise e


# --- Configuration for this Step ---
# It's good practice to make names unique or identifiable
UNIQUE_PREFIX = "my-django-deploy" # <--- CHANGE this to something unique
DO_REGION = "nyc3"  # <--- CHOOSE your desired region (e.g., nyc1, sfo3, lon1)
K8S_CLUSTER_NAME = f"{UNIQUE_PREFIX}-k8s-cluster"
NODE_POOL_NAME = f"{UNIQUE_PREFIX}-worker-pool"
NODE_SIZE = "s-1vcpu-2gb"  # <--- CHOOSE desired node size (Using s-1vcpu-2gb from your log)
NODE_COUNT = 2 # <--- CHOOSE number of nodes
K8S_VERSION = "latest" # Or specify a version like "1.29.1-do.0"

# --- Logic ---
print(f"Initiating creation of Kubernetes Cluster: {K8S_CLUSTER_NAME}")
print(f"Region: {DO_REGION}, Version: {K8S_VERSION}, Nodes: {NODE_COUNT}x {NODE_SIZE}")

# Make sure HEADERS were loaded correctly
if 'HEADERS' not in locals() or not HEADERS:
    print("ERROR: HEADERS variable not found. Please ensure Cell 1 ran successfully and stored HEADERS.")
    sys.exit("Stopping due to missing HEADERS.") # Stop if essential config is missing

cluster_endpoint = f"{DO_API_BASE_URL}/kubernetes/clusters"

payload = {
    "name": K8S_CLUSTER_NAME,
    "region": DO_REGION,
    "version": K8S_VERSION,
    "node_pools": [
        {
            "size": NODE_SIZE,
            "count": NODE_COUNT,
            "name": NODE_POOL_NAME,
            "tags": [f"{UNIQUE_PREFIX}-k8s-node"] # Optional tag
        }
    ]
    # Add 'ha', 'auto_upgrade', 'tags', 'vpc_uuid' if needed
}

cluster_id = None # Initialize cluster_id
cluster_status = None # Initialize cluster_status

# --- Create Cluster ---
try:
    response = requests.post(cluster_endpoint, headers=HEADERS, json=payload)
    # Call the check_do_response function defined *within this cell*
    result = check_do_response(response, f"Create Kubernetes Cluster ({K8S_CLUSTER_NAME})")

    # Check if creation POST was successful and returned expected data
    if response.ok and result and 'kubernetes_cluster' in result:
        cluster_info = result['kubernetes_cluster']
        cluster_id = cluster_info.get('id')
        cluster_status = cluster_info.get('status', {}).get('state', 'unknown')
        print(f"\nCluster creation initiated successfully.")
        print(f"Cluster ID: {cluster_id}")
        print(f"Initial Status: {cluster_status}")
        %store cluster_id # Store the ID for potential later use

        # --- Wait for Cluster to be Running ---
        # Proceed only if we got a cluster_id and initial status wasn't an immediate error state
        if cluster_id and cluster_status not in ['error', 'invalid', 'deleted', 'deleting']:
            print("\nWaiting for cluster to become active (this can take 5-10 minutes)...")
            while cluster_status != 'running':
                time.sleep(30) # Check every 30 seconds
                try:
                    status_response = requests.get(f"{cluster_endpoint}/{cluster_id}", headers=HEADERS)
                    # Avoid verbose check_do_response during polling unless it fails
                    if status_response.ok:
                        status_result = status_response.json()
                        if status_result and 'kubernetes_cluster' in status_result:
                             current_status = status_result['kubernetes_cluster'].get('status', {}).get('state', 'unknown')
                             print(f"  Current status: {current_status} ({time.strftime('%H:%M:%S')})")
                             if current_status == 'running':
                                 cluster_status = 'running' # Update status to exit loop
                                 print("\nCluster is now running!")
                             elif current_status in ['provisioning', 'degraded']:
                                 cluster_status = current_status # Continue waiting, update status
                             else:
                                 print(f"ERROR: Cluster entered unexpected final state: {current_status}")
                                 cluster_status = current_status # Update status to exit loop
                                 break # Exit loop on unexpected state
                        else:
                            print("WARNING: Could not parse cluster status response JSON.")
                            # Optionally break or continue after a delay
                            time.sleep(10)
                    else:
                         # Use the local helper function for detailed error on failure
                         check_do_response(status_response, f"Get Cluster Status ({cluster_id})")
                         print("WARNING: Failed to retrieve cluster status. Will retry...")
                         time.sleep(10) # Wait longer after an error

                except requests.exceptions.RequestException as net_err:
                     # Handle network errors during polling
                     print(f"WARNING: Network error checking cluster status: {net_err}. Retrying...")
                     time.sleep(10) # Wait longer after a network error
                except Exception as poll_err:
                     # Handle other unexpected errors during polling
                     print(f"ERROR: Unexpected error during status poll: {type(poll_err).__name__} - {poll_err}")
                     print("Stopping status check.")
                     cluster_status = "poll_error" # Set status to indicate polling failed
                     break # Exit polling loop

            # --- Check final status after loop ---
            if cluster_status == 'running':
                 print("\n--- ACTION REQUIRED (Local Machine) ---")
                 print("1. Install `doctl` on your local machine if you haven't already:")
                 print("   (See: https://docs.digitalocean.com/reference/doctl/how-to/install/)")
                 print("2. Configure `kubectl` locally by running:")
                 print(f"   doctl kubernetes cluster kubeconfig save {K8S_CLUSTER_NAME}")
                 print("3. Verify `kubectl` connection by running:")
                 print("   kubectl get nodes")
                 print("----------------------------------------\n")
            else:
                 print(f"\nERROR: Cluster did not reach 'running' state. Final status: {cluster_status}")
        else:
             # Handle cases where cluster_id was None or initial status was bad
             print(f"\nERROR: Cluster creation might have failed initially or is in error state: {cluster_status}")

    else:
        # Handle cases where the initial POST failed or didn't return expected data
        print("\nERROR: Failed to initiate cluster creation properly or parse the initial response.")
        # Error details should have been printed by check_do_response

# Catch specific HTTP errors from the initial POST or status GET failures handled by check_do_response
except requests.exceptions.HTTPError as http_err:
    print(f"\nAn HTTP error occurred: {http_err}")
    print("Please check the error details above.")
# Catch network errors during the initial POST
except requests.exceptions.RequestException as req_err:
    print(f"\nAn error occurred during the cluster creation network request: {req_err}")
# Catch any other unexpected errors
except Exception as e:
    print(f"\nAn unexpected error occurred in this cell: {type(e).__name__} - {e}")


# Store name for instructions in later cells, regardless of errors in creation process
print(f"Storing K8S_CLUSTER_NAME: {K8S_CLUSTER_NAME}")
%store K8S_CLUSTER_NAME

In [ ]:
# @title Cell 4: Step 2 - Create DigitalOcean Container Registry

# --- Imports ---
import requests
import json
import time
import sys # Import sys for sys.exit on critical errors

# --- Retrieve stored variables needed here ---
# Ensure variables from previous cells are loaded correctly
# Remove the comment from this line
%store -r DO_API_BASE_URL HEADERS UNIQUE_PREFIX DO_REGION

# --- Helper Function (Copied from Cell 1) ---
# Include the function definition directly in this cell
def check_do_response(response, step_name):
    """Checks DigitalOcean API response status and prints messages."""
    print(f"--- Status Check: {step_name} ---")
    try:
        response.raise_for_status()  # Raises HTTPError for bad responses (4xx or 5xx)
        print(f"SUCCESS: Request successful (Status Code: {response.status_code}).")
        try:
            # Handle cases where success response might be empty (e.g., 204 No Content)
            if response.status_code != 204:
                 return response.json()
            else:
                 print("INFO: Request successful with empty response body (Status 204).")
                 return None
        except json.JSONDecodeError:
            print("WARNING: Response was successful but contained non-JSON data or was empty.")
            return None
    except requests.exceptions.HTTPError as err:
        print(f"ERROR: Request failed.")
        print(f"Status Code: {response.status_code}")
        print(f"Reason: {response.reason}")
        try:
            error_details = response.json()
            print(f"Error Details: {json.dumps(error_details, indent=2)}")
        except json.JSONDecodeError:
            print(f"Response Content: {response.text}")
        # Re-raise the specific HTTPError to be caught by the main try/except
        raise err
    except Exception as e:
        print(f"An unexpected error occurred during response check: {type(e).__name__} - {e}")
        # Re-raise the exception
        raise e


# --- Configuration for this Step ---
# Registry names must be globally unique, lowercase, numbers, hyphens. Cannot exceed 63 chars.
# We still define the desired name, even if one already exists, for potential reference
DO_REGISTRY_NAME_DESIRED = f"{UNIQUE_PREFIX}-registry".lower().replace("_","-")[:63]
DO_REGISTRY_TIER = "starter" # <--- CHOOSE tier: starter, basic, professional

# --- Logic ---
print(f"Checking for existing DigitalOcean Container Registry or creating one (Desired name: {DO_REGISTRY_NAME_DESIRED})")

# Make sure HEADERS were loaded correctly
if 'HEADERS' not in locals() or not HEADERS:
    print("ERROR: HEADERS variable not found. Please ensure Cell 1 ran successfully and stored HEADERS.")
    sys.exit("Stopping due to missing HEADERS.") # Stop if essential config is missing
if 'DO_REGION' not in locals() or not DO_REGION:
     print("ERROR: DO_REGION variable not found. Please ensure Cell 3 ran successfully and stored DO_REGION.")
     sys.exit("Stopping due to missing DO_REGION.") # Stop if essential config is missing

registry_endpoint = f"{DO_API_BASE_URL}/registry"

payload = {
    "name": DO_REGISTRY_NAME_DESIRED,
    "region": DO_REGION, # Use the same region as K8s cluster is often recommended
    "subscription_tier_slug": DO_REGISTRY_TIER
}

registry_full_url = None # Initialize
actual_registry_name = None # Initialize

# --- Create or Get Registry ---
try:
    print(f"Attempting to create registry with name: {DO_REGISTRY_NAME_DESIRED}")
    response = requests.post(registry_endpoint, headers=HEADERS, json=payload)

    # --- Handle Specific Errors ---
    # 422: User already has *a* registry (likely only one allowed per account)
    if response.status_code == 422:
        error_details = response.json()
        if error_details.get('message') == 'user already has a registry':
            print("INFO: Received HTTP 422 - User already has a registry. Attempting to get existing registry details.")
            try:
                # Use GET /v2/registry to find the existing registry
                get_response = requests.get(registry_endpoint, headers=HEADERS)
                registry_info_result = check_do_response(get_response, "Get Existing Registry")
                if registry_info_result and 'registry' in registry_info_result:
                    actual_registry_name = registry_info_result['registry']['name']
                    registry_full_url = f"registry.digitalocean.com/{actual_registry_name}"
                    print(f"SUCCESS: Found existing registry: Name='{actual_registry_name}', URL='{registry_full_url}'")
                else:
                    print("ERROR: Failed to retrieve details of the existing registry even though API indicated one exists.")
                    # Cannot proceed reliably without registry info
                    raise ValueError("Failed to get existing registry details after 422 error.")
            except requests.exceptions.RequestException as get_err:
                 print(f"ERROR: Failed request while trying to GET existing registry info: {get_err}")
                 raise get_err # Raise the error to stop execution
        else:
            # Handle other potential 422 errors if necessary
            print("ERROR: Received unexpected HTTP 422 error.")
            check_do_response(response, f"Create Container Registry ({DO_REGISTRY_NAME_DESIRED})") # Log details
            raise requests.exceptions.HTTPError(response=response) # Raise error

    # 409: Conflict - a registry with this *specific name* already exists (less likely if only one allowed)
    elif response.status_code == 409:
        print(f"INFO: Received HTTP 409 Conflict - Registry name '{DO_REGISTRY_NAME_DESIRED}' might be taken or already yours.")
        # Assume it's yours and try to use the desired name, API might enforce uniqueness anyway
        actual_registry_name = DO_REGISTRY_NAME_DESIRED
        registry_full_url = f"registry.digitalocean.com/{actual_registry_name}"
        print(f"Assuming registry exists with desired name. Using URL: {registry_full_url}")
        # Optionally, add a GET request here like in the 422 block to be certain

    # --- Handle Success (2xx) ---
    elif response.ok:
        # Process normal creation response using the helper function
        result = check_do_response(response, f"Create Container Registry ({DO_REGISTRY_NAME_DESIRED})")
        if result and 'registry' in result:
            registry_info = result['registry']
            actual_registry_name = registry_info.get('name')
            registry_full_url = f"registry.digitalocean.com/{actual_registry_name}"
            print(f"\nContainer Registry '{actual_registry_name}' created successfully.")
            print(f"Registry URL: {registry_full_url}")
            print("Waiting a few seconds for registry to stabilize...")
            time.sleep(10)
        else:
            print("\nERROR: Registry creation request was successful but response format was unexpected.")
            # Fallback assumption if name/URL extraction failed
            actual_registry_name = DO_REGISTRY_NAME_DESIRED
            registry_full_url = f"registry.digitalocean.com/{actual_registry_name}"
            print(f"Assuming registry URL based on desired name: {registry_full_url}")

    # --- Handle Other Errors ---
    else:
        # Use helper to log details for any other non-2xx, non-409, non-422 status
        check_do_response(response, f"Create Container Registry ({DO_REGISTRY_NAME_DESIRED})")
        # Raise error to stop execution
        response.raise_for_status()


    # --- Post-Creation/Get Actions ---
    # Store the determined registry URL and actual name for the next step
    if registry_full_url and actual_registry_name:
        %store registry_full_url actual_registry_name
        print(f"\nStored registry_full_url: {registry_full_url}")
        print(f"Stored actual_registry_name: {actual_registry_name}") # Store the actual name

        print("\n--- ACTION REQUIRED (Local Machine) ---")
        print("1. Ensure Docker is running on your local machine.")
        print("2. Log in to your DigitalOcean Container Registry using `doctl`:")
        print(f"   doctl registry login")
        print("   (Or use: docker login registry.digitalocean.com -u <your-do-email-or-token-name> -p <DO_API_TOKEN>)")
        print("----------------------------------------\n")
    else:
        print("\nCRITICAL ERROR: Could not determine the Container Registry URL. Cannot proceed.")
        sys.exit("Registry URL determination failed.")


# Catch specific HTTP errors that weren't handled above or were re-raised
except requests.exceptions.HTTPError as http_err:
    print(f"\nAn HTTP error occurred that was not handled by specific logic: {http_err}")
    print("Please check the error details above.")
# Catch network errors during the POST/GET requests
except requests.exceptions.RequestException as req_err:
    print(f"\nAn error occurred during a registry network request: {req_err}")
# Catch ValueErrors raised internally (e.g., failed GET after 422)
except ValueError as val_err:
    print(f"\nA configuration or state error occurred: {val_err}")
# Catch any other unexpected errors
except Exception as e:
    print(f"\nAn unexpected error occurred in this cell: {type(e).__name__} - {e}")

# Store the desired name for reference, even if errors occurred
%store DO_REGISTRY_NAME_DESIRED

In [ ]:
# @title Cell 5a: Setup, Variable Loading & API Prep
# -----------------------------------------------------------------------------
# Goal: Install dependencies, load secrets & variables, prepare for GitHub API calls.
# REQUIREMENTS:
#   - Colab Secrets: 'GITHUB_PAT' (repo/workflow scope) & 'DO_PAT' must be set.
#   - Cells 2 & 4 must have run successfully to store non-sensitive variables.
# -----------------------------------------------------------------------------

# --- Installations ---
print("Installing pynacl for secret encryption...")
!pip install pynacl -q

# --- Imports ---
import requests
import json
import base64
import os
import sys
import textwrap
import time
from urllib.parse import urlparse
from nacl.public import PublicKey, SealedBox
from nacl.encoding import Base64Encoder
from google.colab import userdata # Import userdata to access secrets

# --- Retrieve Stored/Secret Variables ---
print("Retrieving sensitive tokens from Colab Secrets...")
variables_loaded = True
github_pat = None
DIGITALOCEAN_TOKEN = None
# !!! IMPORTANT: Set your target repository URL here !!!
GIT_REPO_URL = "https://github.com/your-org/podcast-generator.git" # <--- CHANGE THIS

try:
    # Get sensitive tokens from Colab's Secret Manager
    # Ensure your secrets are named 'GITHUB_PAT' and 'DO_PAT' in Colab
    github_pat = userdata.get('GITHUB_PAT')
    DIGITALOCEAN_TOKEN = userdata.get('DO_API_TOKEN') # Corrected secret name
except Exception as e:
    print(f"Error accessing Colab Secrets: {e}")
    variables_loaded = False

# Check if secrets were loaded successfully
if not github_pat:
    print("ERROR: GitHub PAT not found in Colab Secrets.")
    print("       -> Add secret named 'GITHUB_PAT' via the Key (🔑) icon and grant Notebook access.")
    variables_loaded = False
if not DIGITALOCEAN_TOKEN:
    print("ERROR: DigitalOcean Token not found in Colab Secrets.")
    print("       -> Add secret named 'DO_PAT' via the Key (🔑) icon and grant Notebook access.")
    variables_loaded = False

print("\nRetrieving other variables from storage (%store)...")
actual_registry_name = None
registry_full_url = None
try:
    # Only restore non-sensitive variables using %store from previous cells
    # GIT_REPO_URL is defined above now, no need to restore it.
    %store -r actual_registry_name registry_full_url

    # Check if they actually exist in the local namespace after restore attempt
    if 'actual_registry_name' not in locals() or not locals()['actual_registry_name']: raise KeyError('actual_registry_name')
    if 'registry_full_url' not in locals() or not locals()['registry_full_url']: raise KeyError('registry_full_url')
    # Assign loaded values
    actual_registry_name = locals()['actual_registry_name']
    registry_full_url = locals()['registry_full_url']

except KeyError as e:
    print(f"ERROR: Missing stored variable from previous cells: {e}. Cannot proceed.")
    print("       -> Ensure Cells 2 and 4 ran successfully and stored '%store actual_registry_name' and '%store registry_full_url'.")
    variables_loaded = False
except NameError as e:
    print(f"ERROR: Problem retrieving stored variables: {e}. Did previous cells run?")
    variables_loaded = False

# Final check before proceeding
if not variables_loaded:
    sys.exit("Stopping due to missing variables/secrets. Check setup and previous cell execution.")
else:
    print("Required variables and secrets loaded successfully.")
    print(f"  Registry Name: {actual_registry_name}")
    print(f"  Registry URL: {registry_full_url}")

# --- Configuration for later use ---
# These are less likely to change between runs compared to tokens/URLs
DOCKER_IMAGE_BASE_NAME = "django-app"
WORKFLOW_FILE_PATH = ".github/workflows/build-push.yml"
COMMIT_MESSAGE = "Automated commit: Add/update DO container build workflow"
DO_PAT_SECRET_NAME = "DO_PAT"
DO_REGISTRY_SECRET_NAME = "DO_REGISTRY_NAME"
TRIGGER_BRANCH = "main" # <--- Change if your primary branch is different

# --- GitHub API Setup ---
try:
    parsed_repo_url = urlparse(GIT_REPO_URL)
    path_parts = parsed_repo_url.path.strip('/').split('/')
    if len(path_parts) >= 2:
        GIT_OWNER = path_parts[0]
        GIT_REPO = path_parts[1].replace('.git', '')
        print(f"\nTarget GitHub Repo Parsed: {GIT_OWNER}/{GIT_REPO}")
    else:
        raise ValueError("Could not parse owner/repo from GIT_REPO_URL")
except Exception as e:
    sys.exit(f"ERROR parsing GitHub URL '{GIT_REPO_URL}': {e}")

GITHUB_API_BASE = "https://api.github.com"
GITHUB_HEADERS = {
    "Accept": "application/vnd.github.v3+json",
    "Authorization": f"token {github_pat}" # Uses PAT loaded from Secrets
}

# --- Helper Functions for Secret Encryption & Creation (Needed for Cell 5c) ---
# Define these here as they depend on imports and API base/headers

def encrypt_secret(public_key_b64: str, secret_value: str) -> str:
    """Encrypts a secret string using a public key retrieved from GitHub."""
    try:
        public_key = PublicKey(base64.b64decode(public_key_b64), Base64Encoder)
        sealed_box = SealedBox(public_key)
        encrypted = sealed_box.encrypt(secret_value.encode("utf-8"))
        return base64.b64encode(encrypted).decode("utf-8")
    except Exception as e:
        print(f"  Error during encryption: {e}")
        raise # Re-raise the exception to be caught by caller

def get_github_public_key(owner, repo, headers):
     """Gets the public key required for encrypting Actions secrets."""
     key_url = f"{GITHUB_API_BASE}/repos/{owner}/{repo}/actions/secrets/public-key"
     print("  Getting repository public key for encryption...")
     try:
        key_response = requests.get(key_url, headers=headers)
        key_response.raise_for_status() # Check for HTTP errors (like 404 Not Found)
        key_data = key_response.json()
        key_id = key_data.get("key_id")
        public_key_b64 = key_data.get("key")
        if not key_id or not public_key_b64:
            raise ValueError("Could not retrieve valid public key or key_id from GitHub.")
        print("  Successfully retrieved public key.")
        return key_id, public_key_b64
     except requests.exceptions.HTTPError as err:
        print(f"ERROR getting public key. Status: {err.response.status_code}")
        print(f"Response: {err.response.text}")
        if err.response.status_code == 404:
             print("  Hint: Repo not found, PAT lacks access, or Actions not enabled?")
        elif err.response.status_code == 403:
             print("  Hint: Check PAT permissions for Actions secrets ('repo', 'workflow').")
        return None, None # Indicate failure
     except (requests.exceptions.RequestException, ValueError, KeyError, json.JSONDecodeError) as e:
        print(f"ERROR getting public key: {e}")
        return None, None # Indicate failure

def create_or_update_github_secret(owner, repo, secret_name, secret_value, key_id, public_key_b64, headers):
    """Creates or updates a GitHub Actions secret (assumes key_id/pub_key are valid)."""
    print(f"\nAttempting to create/update secret via API: {secret_name}")
    # 1. Encrypt the secret value
    print(f"  Encrypting secret value for '{secret_name}'...")
    try:
        encrypted_value = encrypt_secret(public_key_b64, secret_value)
    except Exception as e:
        print(f"ERROR encrypting secret '{secret_name}': {e}")
        return False # Cannot proceed

    # 2. Send the encrypted secret to GitHub
    secret_url = f"{GITHUB_API_BASE}/repos/{owner}/{repo}/actions/secrets/{secret_name}"
    payload = {"encrypted_value": encrypted_value, "key_id": key_id}
    print(f"  Sending encrypted secret '{secret_name}' to GitHub...")
    try:
        secret_response = requests.put(secret_url, headers=headers, json=payload)
        secret_response.raise_for_status()
        if secret_response.status_code == 201: print(f"SUCCESS: Secret '{secret_name}' created.")
        elif secret_response.status_code == 204: print(f"SUCCESS: Secret '{secret_name}' updated.")
        else: print(f"SUCCESS: Secret '{secret_name}' op status {secret_response.status_code}.")
        return True
    except requests.exceptions.HTTPError as err:
        print(f"ERROR creating/updating secret '{secret_name}'. Status: {err.response.status_code}")
        print(f"Response: {err.response.text}")
        if err.response.status_code == 403: print("  Hint: Check PAT permissions.")
        elif err.response.status_code == 422: print("  Hint: Secret name/value invalid?")
        return False
    except Exception as e:
        print(f"ERROR during secret API call '{secret_name}': {type(e).__name__} - {e}")
        return False

# --- Store necessary variables for subsequent cells ---
# Store owner/repo info, headers, tokens, registry name for use in 5b and 5c
print("\nStoring configuration for next steps...")
%store GIT_OWNER GIT_REPO GITHUB_HEADERS DIGITALOCEAN_TOKEN actual_registry_name
%store DOCKER_IMAGE_BASE_NAME WORKFLOW_FILE_PATH COMMIT_MESSAGE TRIGGER_BRANCH
%store DO_PAT_SECRET_NAME DO_REGISTRY_SECRET_NAME registry_full_url

print("\nCell 5a finished. Setup and variable loading complete.")
print("Check output above for any errors before proceeding.")

In [ ]:
# @title Cell 5b: Create/Update GitHub Workflow File (Revised for CD - CORRECTED)
# -----------------------------------------------------------------------------
# Goal: Create or update the Actions workflow YAML file in the target repo,
#       now including a basic Continuous Deployment (CD) step.
# Requires: Variables stored successfully by Cell 5a (or defined in the environment).
#           User needs to manually create DO_PAT secret in GitHub.
# -----------------------------------------------------------------------------

# --- Imports ---
import requests
import json
import base64
import os
import sys
import textwrap
import time

# --- Retrieve Stored Variables ---
# Note: Assumes these variables exist in the notebook environment,
# either via %store from Cell 5a or defined manually.
print("Retrieving variables...")
variables_loaded = True
required_vars = [
    'GIT_OWNER', 'GIT_REPO', 'GITHUB_HEADERS', 'TRIGGER_BRANCH',
    'WORKFLOW_FILE_PATH', 'COMMIT_MESSAGE', 'DOCKER_IMAGE_BASE_NAME',
    'registry_full_url', 'K8S_CLUSTER_NAME' # Added K8S_CLUSTER_NAME
]
missing_vars = []

for var_name in required_vars:
    if var_name not in locals():
        missing_vars.append(var_name)
        variables_loaded = False

if not variables_loaded:
    print(f"ERROR: Missing required variables: {', '.join(missing_vars)}. Cannot proceed.")
    print("Ensure these variables were set and stored (e.g., in Cell 5a) or defined.")
    sys.exit("Stopping: Essential variables missing.")
else:
    # Define derived APP_NAME, DEPLOYMENT_NAME consistently
    APP_NAME = DOCKER_IMAGE_BASE_NAME # Use base name for app label/container name
    DEPLOYMENT_NAME = APP_NAME # Use the same for the K8s deployment object name
    print(f"Variables loaded (GIT_OWNER={GIT_OWNER}, GIT_REPO={GIT_REPO}, K8S_CLUSTER_NAME={K8S_CLUSTER_NAME}, etc.).")

# --- GitHub API Setup ---
GITHUB_API_BASE = "https://api.github.com" # Define or ensure it's loaded

# --- Helper Functions for GitHub File API ---
# (Keep helper functions get_github_file_sha and create_or_update_github_file as before)
def get_github_file_sha(owner, repo, path, headers):
    """Gets the SHA hash of an existing file in a GitHub repo."""
    url = f"{GITHUB_API_BASE}/repos/{owner}/{repo}/contents/{path}?ref={TRIGGER_BRANCH}"
    print(f"  Checking if file '{path}' exists on branch '{TRIGGER_BRANCH}'...")
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            sha = response.json().get("sha"); print(f"  File '{path}' found (SHA: {sha[:7]}...)."); return sha
        elif response.status_code == 404: print(f"  File '{path}' not found."); return None
        else: print(f"  Warning: Error checking file ({path}): {response.status_code} {response.text}"); return None
    except requests.exceptions.RequestException as e: print(f"  Warning: Network error checking file ({path}): {e}"); return None

def create_or_update_github_file(owner, repo, path, content_str, commit_msg, headers):
    """Creates or updates a file in the GitHub repository using the API."""
    print(f"\nAttempting to create/update file via API: {path}")
    existing_sha = get_github_file_sha(owner, repo, path, headers)
    url = f"{GITHUB_API_BASE}/repos/{owner}/{repo}/contents/{path}"
    encoded_content = base64.b64encode(content_str.encode('utf-8')).decode('utf-8')
    payload = {"message": commit_msg, "content": encoded_content, "branch": TRIGGER_BRANCH}
    if existing_sha: payload["sha"] = existing_sha; print(f"  Updating existing file '{path}'...")
    else: print(f"  Creating new file '{path}'...")
    try:
        response = requests.put(url, headers=headers, json=payload); response.raise_for_status()
        commit_sha = response.json().get('commit', {}).get('sha', 'N/A')
        print(f"SUCCESS: File '{path}' created/updated. Commit: {commit_sha[:7]}"); time.sleep(3); return True
    except requests.exceptions.HTTPError as err:
        print(f"ERROR creating/updating file '{path}'. Status: {err.response.status_code}"); print(f"Response: {err.response.text}")
        if err.response.status_code == 403: print("  Hint: Check PAT 'repo' scope.")
        elif err.response.status_code == 409: print(f"  Hint: Conflict - branch protected or SHA mismatch?")
        elif err.response.status_code == 404: print(f"  Hint: Not Found - Does the base path '.github/workflows' exist in your repo on branch '{TRIGGER_BRANCH}'?")
        elif err.response.status_code == 422: print(f"  Hint: Unprocessable Entity - path or content invalid?")
        return False
    except Exception as e: print(f"ERROR during file operation '{path}': {type(e).__name__} - {e}"); return False


# --- Main Execution ---
# 1. Define Workflow Content (with CD step)
print("\nDefining Workflow YAML content (including CD step)...")
# Use the stored registry_full_url correctly
image_name_no_tag = f"{registry_full_url}/{DOCKER_IMAGE_BASE_NAME}" # e.g. registry.digitalocean.com/my-reg/django-app

github_action_yaml = textwrap.dedent(f"""\
# Auto-generated by Colab script (with CD step)
name: Build, Push Docker Image, and Deploy to DOKS

on:
  push:
    branches: [ {TRIGGER_BRANCH} ] # Trigger on push to main/master
  workflow_dispatch: # Allow manual trigger

jobs:
  build-push-deploy:
    runs-on: ubuntu-latest
    permissions:
      contents: read
      packages: write # Needed for docker/login-action with DO registry token

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Docker Buildx
        uses: docker/setup-buildx-action@v3

      - name: Log in to DigitalOcean Container Registry
        uses: docker/login-action@v3
        with:
          registry: registry.digitalocean.com
          # Use DO PAT stored as GitHub Secret 'DO_PAT'
          username: ${{{{ secrets.DO_PAT }}}}
          password: ${{{{ secrets.DO_PAT }}}}

      - name: Build and push Docker image
        id: build_push # Give this step an id to reference its outputs
        uses: docker/build-push-action@v5
        with:
          context: .
          file: ./Dockerfile # Assumes Dockerfile is in the root
          push: true
          # Tags: Use commit SHA for specific version and 'latest'
          tags: |
            {image_name_no_tag}:${{{{ github.sha }}}}
            {image_name_no_tag}:latest
          # Optional: Add build args, labels etc. here

      # --- NEW: Continuous Deployment Step ---
      - name: Install doctl
        uses: digitalocean/action-doctl@v2
        with:
          token: ${{{{ secrets.DO_PAT }}}} # Use the same DO PAT

      - name: Save Kubeconfig
        # Use doctl to get kubeconfig for the target cluster
        # IMPORTANT: Assumes K8S_CLUSTER_NAME ({K8S_CLUSTER_NAME}) is the correct name matching your cluster.
        run: doctl kubernetes cluster kubeconfig save {K8S_CLUSTER_NAME} # Use cluster name variable

      - name: Deploy to DOKS using kubectl
        # This step uses the fetched kubeconfig to interact with the cluster
        run: |
          echo "Updating Kubernetes deployment..."
          # Rollout the new image using the 'latest' tag pushed earlier
          # Assumes deployment name matches APP_NAME/DOCKER_IMAGE_BASE_NAME ({DEPLOYMENT_NAME})
          # Assumes container name inside deployment also matches APP_NAME ({APP_NAME})
          # Assumes deployment is in the 'default' namespace, add -n <namespace> if needed
          kubectl set image deployment/{DEPLOYMENT_NAME} {APP_NAME}={image_name_no_tag}:latest

          # Optional: Wait for rollout to complete (can add timeout)
          echo "Waiting for deployment rollout to finish..."
          kubectl rollout status deployment/{DEPLOYMENT_NAME} --timeout=2m

        # Note: KUBECONFIG env var is automatically picked up by kubectl if doctl saved it to default location (~/.kube/config)

# --- IMPORTANT NOTE ---
# This CD step assumes:
# 1. Your Kubernetes Deployment is named '{DEPLOYMENT_NAME}'.
# 2. The container within the Deployment to update is named '{APP_NAME}'.
# 3. You are deploying to the 'default' namespace.
# 4. The necessary RBAC permissions exist for the token used by doctl/kubectl.
# 5. The DigitalOcean Personal Access Token (DO_PAT secret) has the necessary permissions to access the Kubernetes cluster details via doctl.
""")
# print(github_action_yaml) # Optional: print the yaml

# 2. Create/Update Workflow File
print("\nAttempting to commit workflow file (with CD) to GitHub...")
file_ok = create_or_update_github_file(
    GIT_OWNER, GIT_REPO, WORKFLOW_FILE_PATH, github_action_yaml, COMMIT_MESSAGE + " (with CD step)", GITHUB_HEADERS
)

# Store the result for the next cell (or just use the variable directly)
# %store file_ok # Uncomment if using %store magic

if file_ok:
    print("\nCell 5b (Revised & Corrected) finished successfully.")
    print(f"Check your repository '{GIT_OWNER}/{GIT_REPO}' for the updated file '{WORKFLOW_FILE_PATH}'.")
    print("\nNEXT STEP: Ensure the required secrets (DO_PAT) are correctly configured in your GitHub repository settings.")
    # print("See Cell 5c (for DO_PAT/DO_REGISTRY_NAME) and potentially Cell 5.6 (for KUBE_CONFIG_DATA, though this version uses doctl).") # Adjust as needed

else:
    print("\nCell 5b (Revised & Corrected) finished with errors.")
    print("Workflow file was not created/updated.")
    print("Please review errors above.")

In [ ]:
# @title Cell 5c: Create/Update GitHub Secrets & Trigger Workflow
# -----------------------------------------------------------------------------
# Goal: Create/update required Actions secrets (DO_PAT, DO_REGISTRY_NAME) and
#       optionally trigger the workflow.
# Requires: Variables stored successfully by Cell 5a. Success status from 5b.
# -----------------------------------------------------------------------------

# --- Imports ---
import requests
import json
import base64
import os
import sys
import textwrap
import time
from urllib.parse import urlparse
# Import nacl specific components for encryption
from nacl.public import PublicKey, SealedBox
from nacl.encoding import Base64Encoder
import traceback # Import traceback for detailed error printing

# --- Retrieve Stored Variables ---
print("Retrieving variables from Cell 5a...")
variables_loaded = True
try:
    # Load variables needed for secret creation and triggering
    %store -r GIT_OWNER GIT_REPO GITHUB_HEADERS DIGITALOCEAN_TOKEN actual_registry_name
    %store -r DO_PAT_SECRET_NAME DO_REGISTRY_SECRET_NAME WORKFLOW_FILE_PATH TRIGGER_BRANCH
    # Load status from previous cell
    %store -r file_ok
    # Load variables needed for final image name storage
    %store -r registry_full_url DOCKER_IMAGE_BASE_NAME

except KeyError as e:
    print(f"ERROR: Missing stored variable from Cell 5a/5b: {e}. Cannot proceed.")
    variables_loaded = False
except NameError as e:
     print(f"ERROR: Problem retrieving stored variables: {e}. Did Cell 5a/5b run?")
     variables_loaded = False

if not variables_loaded:
    sys.exit("Stopping: Essential variables missing.")
elif 'file_ok' not in locals():
     # If file_ok wasn't stored (e.g., error in 5b before %store), assume failure for safety
     print("ERROR: Status of file creation ('file_ok') not found. Assuming file creation failed.")
     file_ok = False # Assume failure if status unknown
     # sys.exit("Stopping: Cannot determine workflow file status.") # Option to stop immediately
else:
     print("Variables loaded.")


# --- GitHub API Setup ---
GITHUB_API_BASE = "https://api.github.com"

# --- Helper Functions for GitHub API (Encryption & Secrets) ---
# Define necessary secret helpers here again for independence

def encrypt_secret(public_key_b64: str, secret_value: str) -> str:
    """Encrypts a secret string using a public key retrieved from GitHub."""
    # --- Start Added Debugging ---
    print(f"    DEBUG inside encrypt_secret:") # Add identifier
    print(f"      Input public_key_b64 type: {type(public_key_b64)}")
    # Use repr() which might show hidden characters like newline (\n)
    print(f"      Input public_key_b64 repr: {repr(public_key_b64)}")
    print(f"      Input public_key_b64 length: {len(public_key_b64) if public_key_b64 else 'None'}")
    if isinstance(public_key_b64, str) and len(public_key_b64) > 20:
         # Print first/last chars to confirm it looks like a key
         print(f"      Input public_key_b64 sample: '{public_key_b64[:10]}...{public_key_b64[-10:]}'")
    else:
         print(f"      Input public_key_b64 value: '{public_key_b64}'")
    # --- End Added Debugging ---

    # Basic validity check (might be redundant now but safe)
    if not isinstance(public_key_b64, str) or len(public_key_b64) % 4 != 0:
         print("    ERROR: Key format check failed INSIDE encrypt_secret.") # Added location info
         raise ValueError(f"Invalid Base64 key format passed to encrypt_secret (length {len(public_key_b64)}). Value starts: {repr(public_key_b64[:20])}")

    try:
        # Decode the Base64 encoded public key from GitHub
        print(f"      Attempting base64.b64decode on key with length {len(public_key_b64)}...") # Add before decode
        decoded_key = base64.b64decode(public_key_b64) # <--- This is where the error likely occurs
        print(f"      Decoding successful (decoded length: {len(decoded_key)}).") # Add after decode

        print(f"      Attempting PublicKey creation...") # Add before PublicKey
        # Pass the *decoded bytes* directly to PublicKey
        public_key = PublicKey(decoded_key)
        print(f"      PublicKey creation successful.") # Add after PublicKey

        # Create a SealedBox object for encryption with the public key
        sealed_box = SealedBox(public_key)
        # Encrypt the secret value (must be bytes)
        print(f"      Attempting sealed_box.encrypt...") # Add before encrypt
        encrypted = sealed_box.encrypt(secret_value.encode("utf-8"))
        print(f"      Encryption successful.") # Add after encrypt

        # Return the Base64 encoded encrypted value
        return base64.b64encode(encrypted).decode("utf-8")
    except Exception as e:
        # Print the specific error during encryption/decoding
        print(f"    ERROR during encryption/decoding step: {type(e).__name__} - {e}")
        # Include traceback info
        print("      Traceback (most recent call last):")
        traceback.print_exc(limit=2, file=sys.stdout) # Limit traceback depth
        raise # Re-raise to be caught by caller

def get_github_public_key(owner, repo, headers):
     """Gets the public key required for encrypting Actions secrets."""
     key_url = f"{GITHUB_API_BASE}/repos/{owner}/{repo}/actions/secrets/public-key";
     print("  Getting repository public key for encryption...")
     key_id, public_key_b64 = None, None # Ensure defined
     try:
        key_response = requests.get(key_url, headers=headers)
        key_response.raise_for_status()
        key_data = key_response.json()
        key_id = key_data.get("key_id")
        public_key_b64 = key_data.get("key")

        # --- Robust Checks ---
        if not isinstance(key_id, str) or not key_id:
            raise ValueError(f"Invalid or missing key_id received: '{key_id}'")
        if not isinstance(public_key_b64, str) or not public_key_b64:
             raise ValueError(f"Invalid or missing public_key (key) received: '{public_key_b64}'")
        if len(public_key_b64) % 4 != 0:
             # This check should catch the base64 format error *before* returning the key
             raise ValueError(f"Retrieved public_key is not valid Base64 (length {len(public_key_b64)} is not multiple of 4). Value: '{public_key_b64[:20]}...'")
        # --- End Checks ---

        # Debug Print (Optional - comment out if too verbose)
        # print(f"  DEBUG: Retrieved public_key_b64 (length: {len(public_key_b64)}): Starts with '{public_key_b64[:10]}...'")

        print("  Successfully retrieved valid public key data.")
        return key_id, public_key_b64
     except requests.exceptions.HTTPError as err:
        print(f"ERROR getting public key. Status: {err.response.status_code}")
        try: print(f"Response: {err.response.json()}")
        except json.JSONDecodeError: print(f"Response: {err.response.text}")
        if err.response.status_code == 404: print("  Hint: Repo not found, PAT lacks access, or Actions not enabled?")
        elif err.response.status_code == 403: print("  Hint: Check PAT permissions.")
        return None, None
     except (requests.exceptions.RequestException, ValueError, KeyError, json.JSONDecodeError) as e:
        print(f"ERROR processing public key response: {e}")
        return None, None
     except Exception as e:
        print(f"Unexpected ERROR getting public key: {type(e).__name__} - {e}")
        return None, None

def create_or_update_github_secret(owner, repo, secret_name, secret_value, key_id, public_key_b64, headers):
    """Creates or updates a GitHub Actions secret (assumes key_id/pub_key are valid)."""
    # Check if key info is valid before proceeding
    if not key_id or not public_key_b64:
         print(f"ERROR: Cannot create/update secret '{secret_name}' due to missing/invalid public key info.")
         return False

    print(f"\nAttempting to create/update secret via API: {secret_name}")
    print(f"  Encrypting secret value for '{secret_name}'...")
    try:
        # Encrypt using the provided key/value
        encrypted_value = encrypt_secret(public_key_b64, secret_value)
    except Exception as e:
        # Error details printed inside encrypt_secret
        print(f"Halting secret creation for '{secret_name}' due to encryption error.")
        return False # Cannot proceed

    # Proceed only if encryption succeeded
    secret_url = f"{GITHUB_API_BASE}/repos/{owner}/{repo}/actions/secrets/{secret_name}"
    payload = {"encrypted_value": encrypted_value, "key_id": key_id}
    print(f"  Sending encrypted secret '{secret_name}' to GitHub...")
    try:
        secret_response = requests.put(secret_url, headers=headers, json=payload)
        secret_response.raise_for_status()
        if secret_response.status_code == 201: print(f"SUCCESS: Secret '{secret_name}' created.")
        elif secret_response.status_code == 204: print(f"SUCCESS: Secret '{secret_name}' updated.")
        else: print(f"SUCCESS: Secret '{secret_name}' op status {secret_response.status_code}.")
        return True
    except requests.exceptions.HTTPError as err:
        print(f"ERROR creating/updating secret '{secret_name}'. Status: {err.response.status_code}")
        try: print(f"Response: {err.response.json()}")
        except json.JSONDecodeError: print(f"Response: {err.response.text}")
        if err.response.status_code == 403: print("  Hint: Check PAT permissions ('repo', 'workflow').")
        elif err.response.status_code == 422: print("  Hint: Secret name/value invalid?")
        return False
    except Exception as e:
        print(f"ERROR during secret API call '{secret_name}': {type(e).__name__} - {e}")
        return False

def trigger_workflow(owner, repo, workflow_filename, branch, headers):
     """Triggers a GitHub Actions workflow that has 'on: workflow_dispatch:'."""
     print(f"\nAttempting to trigger workflow via API: {workflow_filename} on branch {branch}")
     url = f"{GITHUB_API_BASE}/repos/{owner}/{repo}/actions/workflows/{workflow_filename}/dispatches"
     payload = {"ref": branch}
     try:
         response = requests.post(url, headers=headers, json=payload)
         if response.status_code == 204:
             print(f"SUCCESS: Workflow '{workflow_filename}' triggered on branch '{branch}'. Check Actions tab.")
             return True
         else:
             try: print(f"Response: {response.json()}") # Try json first
             except json.JSONDecodeError: print(f"Response: {response.text}") # Fallback to text
             response.raise_for_status() # Raise error after printing details
             return False # Should not be reached
     except requests.exceptions.HTTPError as err:
         print(f"ERROR triggering workflow. Status: {err.response.status_code}")
         # Response details printed above
         if err.response.status_code == 403: print("  Hint: Check PAT permissions.")
         elif err.response.status_code == 404: print(f"  Hint: Workflow '{workflow_filename}' not found or no 'workflow_dispatch' trigger?")
         elif err.response.status_code == 422: print(f"  Hint: Workflow disabled or ref '{branch}' invalid?")
         return False
     except Exception as e:
        print(f"ERROR triggering workflow: {type(e).__name__} - {e}")
        return False

# --- Main Execution ---
print("\n--- Configuring GitHub Actions Secrets ---")

# 1. Get Public Key (needed for both secrets)
key_id, public_key_b64 = get_github_public_key(GIT_OWNER, GIT_REPO, GITHUB_HEADERS)
secrets_ok = False # Flag for overall secret success

# Proceed only if public key was retrieved successfully
if key_id and public_key_b64:
    # 2. Create/Update Secrets
    secret_do_pat_ok = create_or_update_github_secret(
        GIT_OWNER, GIT_REPO, DO_PAT_SECRET_NAME, DIGITALOCEAN_TOKEN, key_id, public_key_b64, GITHUB_HEADERS
    )
    secret_do_reg_ok = create_or_update_github_secret(
        GIT_OWNER, GIT_REPO, DO_REGISTRY_SECRET_NAME, actual_registry_name, key_id, public_key_b64, GITHUB_HEADERS
    )
    # Update overall flag based on *both* secrets succeeding
    secrets_ok = secret_do_pat_ok and secret_do_reg_ok
else:
    print("\nERROR: Failed to get public key. Cannot create/update secrets.")
    # Ensure secrets_ok remains False if key retrieval failed
    secrets_ok = False


# 3. Check Status and Optionally Trigger Workflow
# Check if file creation succeeded (from Cell 5b) AND secret creation succeeded (from this cell)
if file_ok and secrets_ok:
    print("\nSUCCESS: GitHub Actions workflow file and secrets configured via API.")
    trigger_choice = input("Trigger workflow run now? (yes/no): ").strip().lower()
    if trigger_choice == 'yes':
        trigger_ok = trigger_workflow(GIT_OWNER, GIT_REPO, os.path.basename(WORKFLOW_FILE_PATH), TRIGGER_BRANCH, GITHUB_HEADERS)
        if not trigger_ok: print("Workflow trigger failed. Check Actions tab.")
    else: print(f"\nWorkflow not triggered. Push to '{TRIGGER_BRANCH}' or trigger manually.")
# Handle specific failure cases
elif not file_ok:
     print("\nERROR: Workflow file creation failed in previous step (Cell 5b). Secrets not created/updated. Workflow cannot be triggered.")
     sys.exit("Stopping: Workflow file setup failed.")
else: # File was ok, but secrets failed (secrets_ok is False)
    print("\nERROR: Failed to configure GitHub Actions secrets via API.")
    print("Review error messages above, potentially related to public key retrieval or secret encryption/update.")
    sys.exit("Stopping: GitHub Actions secrets setup failure.")


# 4. Define Image Name for Deployment steps
# Assumes the workflow will successfully push the ':latest' tag.
DOCKER_IMAGE_FULL_NAME = f"{registry_full_url}/{DOCKER_IMAGE_BASE_NAME}:latest" # Use registry_full_url loaded earlier
print(f"\nScript will use image name: '{DOCKER_IMAGE_FULL_NAME}'")
print("Ensure the GitHub Action runs successfully before next deployment steps.")
# Store variables needed for Cell 6+
%store DOCKER_IMAGE_FULL_NAME DOCKER_IMAGE_BASE_NAME actual_registry_name

print("\n-----------------------------------------")
print("GitHub Actions setup via API attempted. Verification Steps:")
print("  1. Check repo file:", WORKFLOW_FILE_PATH)
print("  2. Check repo Actions Secrets:", DO_PAT_SECRET_NAME, ",", DO_REGISTRY_SECRET_NAME)
print("  3. Check repo 'Actions' tab for workflow runs.")
print("-----------------------------------------")
print("\nCell 5c finished.")

In [ ]:
# @title Cell 5.5: Step 4 - Create Kubernetes Secret for Django App
# -----------------------------------------------------------------------------
# Goal: Gather sensitive data (like Django Secret Key, Database URL) and
#       generate a Kubernetes Secret manifest. Optionally attempt API creation.
# Requires: Variables from Cell 5a/5c (APP_NAME maybe, or define here).
#           User input for sensitive data.
# Optional: kubectl/doctl setup (from Cell 7.5) for direct API creation.
# -----------------------------------------------------------------------------

# --- Imports ---
import yaml
import sys
import base64
import getpass
# Import kubernetes client library ONLY if attempting automation
# import kubernetes

# --- Retrieve stored variables needed ---
# Let's redefine APP_NAME here based on DOCKER_IMAGE_BASE_NAME for clarity,
# or retrieve if stored reliably earlier.
try:
    # %store -r DOCKER_IMAGE_BASE_NAME # Needed to derive default APP_NAME
    # %store -r K8S_CLUSTER_NAME # Needed for context/instructions
    APP_NAME = DOCKER_IMAGE_BASE_NAME # Use base image name as default app name
    # Check if K8S_CLUSTER_NAME exists
    if 'K8S_CLUSTER_NAME' not in locals(): K8S_CLUSTER_NAME = "<K8S_CLUSTER_NAME_MISSING>"

except (NameError, KeyError) as e:
    print(f"Warning: Could not retrieve stored variables ({e}). Using defaults.")
    if 'APP_NAME' not in locals(): APP_NAME = "django-app" # Fallback app name
    if 'K8S_CLUSTER_NAME' not in locals(): K8S_CLUSTER_NAME = "your-k8s-cluster"

# --- Configuration ---
SECRET_NAME = f"{APP_NAME}-secrets"
# Define keys you want in the secret
# Common examples for Django:
SECRET_KEYS_TO_GATHER = ["DJANGO_SECRET_KEY", "DATABASE_URL"]
# Add other keys as needed, e.g., "EMAIL_HOST_PASSWORD", "STRIPE_SECRET_KEY"

# --- Gather Secret Data ---
print("--- Gathering Sensitive Data for Kubernetes Secret ---")
print(f"You will be prompted to enter values for the Kubernetes secret '{SECRET_NAME}'.")
print("Input will be hidden.")

secret_data = {}
for key in SECRET_KEYS_TO_GATHER:
    value = getpass.getpass(f"Enter value for {key}: ")
    if not value:
        print(f"Warning: No value provided for {key}. It will be empty in the secret.")
        # Decide if this is critical
        # if key == "DJANGO_SECRET_KEY" or key == "DATABASE_URL":
        #    sys.exit(f"ERROR: {key} is required. Exiting.")
    # Kubernetes secrets must be base64 encoded
    secret_data[key] = base64.b64encode(value.encode('utf-8')).decode('utf-8')

# --- Define Secret Manifest ---
secret_manifest = {
    "apiVersion": "v1",
    "kind": "Secret",
    "metadata": {
        "name": SECRET_NAME,
        # Add namespace if not default: "namespace": "your-namespace"
    },
    "type": "Opaque", # Default type for arbitrary key-value pairs
    "data": secret_data
}

secret_file = "secret.yaml"
print(f"\n--- Generated Kubernetes Secret Manifest ({secret_file}) ---")
try:
    secret_yaml_output = yaml.dump(secret_manifest, sort_keys=False)
    print(secret_yaml_output)
    with open(secret_file, 'w') as f:
        f.write(secret_yaml_output)
    print("-"*(len(secret_file)+8))
    print(f"Manifest saved to Colab environment as '{secret_file}'.")
    # %store secret_file SECRET_NAME # Store for reference in Cell 6 and instructions

except yaml.YAMLError as e:
    print(f"Error generating YAML: {e}")
    secret_file = None # Ensure file isn't used if invalid
except IOError as e:
    print(f"Error saving file {secret_file}: {e}")
    secret_file = None # Ensure file isn't used if saving failed

# --- Action Required: Apply the Secret ---
if secret_file: # Only provide instructions if YAML was generated and saved
    print(f"\n--- ACTION REQUIRED (Local Machine - using kubectl configured for {K8S_CLUSTER_NAME}) ---")
    print(f"1. Download the generated '{secret_file}' from Colab's file browser.")
    print(f"2. Apply the secret to your cluster:")
    print(f"   kubectl apply -f {secret_file}")
    print(f"3. Verify creation (optional):")
    print(f"   kubectl get secret {SECRET_NAME}")
    print("------------------------------------------------------------------------------------\n")
else:
    print("\nERROR: Could not generate or save the Secret manifest. Please check errors above.")

# --- Optional: Attempt Direct API Creation (Requires Cell 7.5 setup) ---
# attempt_api_creation = False # Set to True only if Cell 7.5 is implemented and working
# if attempt_api_creation:
#     print("\n--- Attempting Direct Secret Creation via Kubernetes API ---")
#     try:
#         # Ensure kubernetes client is imported and configured (logic from Cell 7.5)
#         # load_kube_config() or similar setup needed here
#         core_v1_api = kubernetes.client.CoreV1Api()
#         namespace = "default" # Or get from config/user
#         try:
#             # Try to patch if it exists (idempotent-like)
#             core_v1_api.patch_namespaced_secret(SECRET_NAME, namespace, secret_manifest)
#             print(f"Patched existing secret '{SECRET_NAME}' in namespace '{namespace}'.")
#         except kubernetes.client.ApiException as e:
#             if e.status == 404:
#                 # Create if it doesn't exist
#                 core_v1_api.create_namespaced_secret(namespace, secret_manifest)
#                 print(f"Created new secret '{SECRET_NAME}' in namespace '{namespace}'.")
#             else:
#                 # Re-raise other API errors
#                 raise e
#         print("Secret creation/update via API successful.")
#     except NameError:
#         print("Skipping API creation: 'kubernetes' client library not loaded.")
#         print("Ensure optional Cell 7.5 has been run and configured if you want API automation.")
#     except Exception as e:
#         print(f"ERROR creating/updating secret via API: {e}")
#         print("Please apply manually using the generated YAML file.")

print("Cell 5.5 finished.")

In [ ]:
# @title Cell 5.6: Step 4c (Manual Action) - Instructions for Kubeconfig GitHub Secret (If Needed)
# -----------------------------------------------------------------------------
# NOTE: The revised workflow in Cell 5b attempts to use `doctl` directly within
#       the GitHub Action to fetch the kubeconfig, avoiding the need for a
#       separate KUBE_CONFIG_DATA secret if the DO_PAT has sufficient permissions.
#
#       This cell provides instructions ONLY IF you modify the workflow's CD
#       step to require a pre-existing kubeconfig passed via a secret named
#       `KUBE_CONFIG_DATA`.
#
# Goal: Explain how to manually generate, encode, and store the kubeconfig
#       needed by the GitHub Actions CD step AS A GITHUB SECRET.
# Requires: User has `doctl` and `kubectl` configured locally. K8S_CLUSTER_NAME.
# -----------------------------------------------------------------------------

import base64

# --- Retrieve variables ---
try:
    # %store -r K8S_CLUSTER_NAME
    if 'K8S_CLUSTER_NAME' not in locals(): K8S_CLUSTER_NAME = "<K8S_CLUSTER_NAME_MISSING>"
except (NameError, KeyError):
    K8S_CLUSTER_NAME = "your-k8s-cluster" # Fallback

# --- Configuration ---
TARGET_SECRET_NAME = "KUBE_CONFIG_DATA" # Name the workflow expects

# --- Instructions ---
print(f"--- MANUAL ACTION REQUIRED (Local Machine) ---")
print(f"IF your GitHub Action's deploy step needs a pre-configured kubeconfig")
print(f"passed via a secret named '{TARGET_SECRET_NAME}', follow these steps:")
print("(The current workflow in Cell 5b tries using `doctl` directly, so this might not be needed).")

if K8S_CLUSTER_NAME == "<K8S_CLUSTER_NAME_MISSING>":
    print("\nERROR: K8S_CLUSTER_NAME missing. Cannot provide accurate doctl command.")
else:
    print(f"\n1. Fetch Kubeconfig Locally:")
    print(f"   Run this command on your local machine where doctl is configured:")
    print(f"   doctl kubernetes cluster kubeconfig save {K8S_CLUSTER_NAME}")
    print(f"   This usually saves/updates the file at '~/.kube/config'.")

    print(f"\n2. Base64 Encode the Kubeconfig:")
    print(f"   Run this command (adjust path if needed):")
    print(f"   cat ~/.kube/config | base64")
    print(f"   On Windows (PowerShell):")
    print(f"   [Convert]::ToBase64String([System.Text.Encoding]::UTF8.GetBytes((Get-Content -Path $env:USERPROFILE\\.kube\\config -Raw)))")
    print(f"   On Windows (Git Bash/WSL):")
    print(f"   cat ~/.kube/config | base64 -w0")

    print(f"\n3. Create GitHub Actions Secret:")
    print(f"   a. Go to your GitHub repository -> Settings -> Secrets and variables -> Actions.")
    print(f"   b. Click 'New repository secret'.")
    print(f"   c. Enter the name: {TARGET_SECRET_NAME}")
    print(f"   d. Paste the ENTIRE base64 encoded output from step 2 into the 'Secret' value box.")
    print(f"   e. Click 'Add secret'.")

print(f"\nOnce '{TARGET_SECRET_NAME}' is created, the GitHub Action step that needs it")
print(f"(e.g., using `actions/setup-kubectl` or directly setting KUBECONFIG env var)")
print(f"can reference it using `${{{{ secrets.{TARGET_SECRET_NAME} }}}}`.")
print("------------------------------------------------")

print("\nCell 5.6 finished providing instructions.")

In [ ]:
# @title Cell 6: Step 5 (Part 1) - Generate Kubernetes Manifests (Revised)

# --- Imports ---
import yaml
import sys

# --- Retrieve stored variables needed ---
# Use the image name generated/stored in Cell 5c
# Retrieve Secret name generated in Cell 5.5
try:
     pass  # Add this line, indented with 4 spaces
     # %store -r DOCKER_IMAGE_FULL_NAME DOCKER_IMAGE_BASE_NAME
     # %store -r secret_file SECRET_NAME # Get secret name
except (NameError, KeyError) as e:
    print(f"Warning: Could not retrieve stored variables ({e}). Using defaults.")
    if 'DOCKER_IMAGE_FULL_NAME' not in locals(): DOCKER_IMAGE_FULL_NAME = "your-registry/your-image:latest"
    if 'DOCKER_IMAGE_BASE_NAME' not in locals(): DOCKER_IMAGE_BASE_NAME = "django-app"
    if 'SECRET_NAME' not in locals(): SECRET_NAME = f"{DOCKER_IMAGE_BASE_NAME}-secrets" # Guess secret name


# --- Configuration for this Step ---
APP_NAME = DOCKER_IMAGE_BASE_NAME # Kubernetes app name/label
DJANGO_PROJECT_NAME = "backend" # <--- The name of your Django project directory (containing settings.py, wsgi.py)
REPLICAS = 2 # <--- Number of pods for your application
# --- NEW: Health Check Config ---
READINESS_PROBE_PATH = "/healthy" # <--- CHANGE if you have a specific health check URL
LIVENESS_PROBE_PATH = "/healthy"  # <--- CHANGE if you have a specific health check URL
PROBE_INITIAL_DELAY = 15 # Seconds to wait before first probe
PROBE_PERIOD = 20       # Seconds between probes
PROBE_TIMEOUT = 5        # Seconds to wait for probe response

# --- Logic ---

# --- Confirm/Correct Docker Image Name ---
print(f"The script currently assumes the full Docker image name is:")
print(f"  '{DOCKER_IMAGE_FULL_NAME}'")
# (Keep user confirmation logic as before...)
correct_image_name = input("Press Enter if this is correct, or paste the exact correct image name now: ").strip()
if correct_image_name:
    DOCKER_IMAGE_FULL_NAME = correct_image_name
    print(f"Using updated image name: {DOCKER_IMAGE_FULL_NAME}")
    # %store DOCKER_IMAGE_FULL_NAME # Store the corrected name
elif not DOCKER_IMAGE_FULL_NAME or DOCKER_IMAGE_FULL_NAME == "your-registry/your-image:latest":
    print("ERROR: Docker image name is missing or using placeholder.")
    sys.exit("Cannot generate manifests without a valid Docker image name.") # Stop execution

# --- Gunicorn/WSGI Assumption ---
print("\n--- Important Assumption: Gunicorn/WSGI Server ---")
print("This deployment assumes your Docker container runs a WSGI server (like Gunicorn)")
print("listening on port 8000 (0.0.0.0:8000).")
print("Example Dockerfile CMD (adjust project/app name and worker count):")
print('  CMD ["gunicorn", "--bind", "0.0.0.0:8000", "--workers=3", "your_django_project.wsgi:application"]')
print("Ensure your Dockerfile's CMD or ENTRYPOINT is correctly configured.")
print("-------------------------------------------------\n")

# --- Define Deployment YAML (Revised) ---
deployment_manifest = {
    "apiVersion": "apps/v1",
    "kind": "Deployment",
    "metadata": {"name": APP_NAME, "labels": {"app": APP_NAME}},
    "spec": {
        "replicas": REPLICAS,
        "selector": {"matchLabels": {"app": APP_NAME}},
        "template": {
            "metadata": {"labels": {"app": APP_NAME}},
            "spec": {
                "containers": [{
                    "name": APP_NAME,
                    "image": DOCKER_IMAGE_FULL_NAME,
                    "ports": [{"containerPort": 8000}], # Port Gunicorn runs on

                    # --- MODIFIED: Use envFrom to load all keys from the secret ---
                    "envFrom": [
                        {"secretRef": {"name": SECRET_NAME}}
                    ],
                    # --- You can still add non-secret ENV vars here if needed ---
                    "env": [
                         {"name": "DJANGO_SETTINGS_MODULE", "value": f"{DJANGO_PROJECT_NAME}.settings"},
                         {"name": "PYTHONUNBUFFERED", "value": "1"}, # Good practice for logging
                         # Add other non-sensitive config, or use ConfigMaps
                    ],

                    # --- NEW: Liveness Probe ---
                    "livenessProbe": {
                        "httpGet": {
                            "path": LIVENESS_PROBE_PATH,
                            "port": 8000 # Port inside the container
                        },
                        "initialDelaySeconds": PROBE_INITIAL_DELAY + 15, # Give more time than readiness
                        "periodSeconds": PROBE_PERIOD + 10, # Check less often than readiness maybe
                        "timeoutSeconds": PROBE_TIMEOUT,
                        "failureThreshold": 3,
                    },
                    # --- NEW: Readiness Probe ---
                    "readinessProbe": {
                        "httpGet": {
                            "path": READINESS_PROBE_PATH,
                            "port": 8000 # Port inside the container
                        },
                        "initialDelaySeconds": PROBE_INITIAL_DELAY,
                        "periodSeconds": PROBE_PERIOD,
                        "timeoutSeconds": PROBE_TIMEOUT,
                        "successThreshold": 1,
                        "failureThreshold": 3,
                    },
                    # --- Optional: Add Resource Requests/Limits ---
                    # "resources": {
                    #    "requests": {
                    #        "memory": "128Mi",
                    #        "cpu": "100m" # 0.1 vCPU
                    #    },
                    #    "limits": {
                    #        "memory": "512Mi",
                    #        "cpu": "500m" # 0.5 vCPU
                    #    }
                    # }
                }],
                # --- Reference registry secret if needed (Not strictly required for DOCR if using DO k8s integration) ---
                # Add imagePullSecrets if your DOKS cluster isn't automatically configured
                # for your DO Container Registry. This usually involves creating a specific
                # secret type `kubernetes.io/dockerconfigjson`.
                # See: https://docs.digitalocean.com/products/kubernetes/how-to/use-container-registry/
                # "imagePullSecrets": [{"name": "your-registry-secret-name"}]
            }
        }
    }
}

# --- Define Service YAML (Unchanged from original) ---
SERVICE_NAME = f"{APP_NAME}-service"
service_manifest = {
    "apiVersion": "v1",
    "kind": "Service",
    "metadata": {
        "name": SERVICE_NAME,
        "labels": {"app": APP_NAME},
        # Add DO Load Balancer annotations if needed
        # "annotations": {"service.beta.kubernetes.io/do-loadbalancer-protocol": "http", ...}
    },
    "spec": {
        "type": "LoadBalancer",
        "selector": {"app": APP_NAME},
        "ports": [{"protocol": "TCP", "port": 80, "targetPort": 8000}] # External 80 -> Internal 8000
    }
}

# --- Save/Output Manifests ---
deployment_file = "deployment.yaml"
service_file = "service.yaml"

print(f"\n--- {deployment_file} (Revised) ---")
try:
    deployment_yaml_output = yaml.dump(deployment_manifest, sort_keys=False)
    print(deployment_yaml_output)
    with open(deployment_file, 'w') as f:
        f.write(deployment_yaml_output)
    print("--------------------------------")
except yaml.YAMLError as e:
    print(f"Error generating deployment YAML: {e}")
    deployment_file = None
except IOError as e:
    print(f"Error saving file {deployment_file}: {e}")
    deployment_file = None


print(f"\n--- {service_file} ---")
try:
    service_yaml_output = yaml.dump(service_manifest, sort_keys=False)
    print(service_yaml_output)
    with open(service_file, 'w') as f:
        f.write(service_yaml_output)
    print("----------------------")
except yaml.YAMLError as e:
    print(f"Error generating service YAML: {e}")
    service_file = None
except IOError as e:
    print(f"Error saving file {service_file}: {e}")
    service_file = None


if deployment_file and service_file:
    print(f"\nManifests generated/updated and saved to Colab environment as '{deployment_file}' and '{service_file}'.")
    # Store names needed for next steps
    # %store deployment_file service_file APP_NAME SERVICE_NAME
else:
    print("\nERROR: Failed to generate or save one or both manifests. Check errors.")

In [ ]:
# @title Cell 7: Step 5 (Part 2) - Apply Kubernetes Manifests (Local Instructions)

# --- Retrieve stored variables needed for instructions ---
# Check if variables exist before trying to restore, provide defaults or error if missing
try:
    %store -r deployment_file service_file APP_NAME SERVICE_NAME K8S_CLUSTER_NAME
    # Check if they actually loaded
    if 'K8S_CLUSTER_NAME' not in locals(): K8S_CLUSTER_NAME = "<K8S_CLUSTER_NAME_MISSING>"
    if 'deployment_file' not in locals(): deployment_file = "deployment.yaml"
    if 'service_file' not in locals(): service_file = "service.yaml"
    if 'APP_NAME' not in locals(): APP_NAME = "<APP_NAME_MISSING>"
    if 'SERVICE_NAME' not in locals(): SERVICE_NAME = "<SERVICE_NAME_MISSING>"

except KeyError:
    print("Warning: Some variables from previous steps were not found in storage. Using defaults.")
    # Define defaults if restore fails - replace with actual expected values if possible
    if 'K8S_CLUSTER_NAME' not in locals(): K8S_CLUSTER_NAME = "your-k8s-cluster-name"
    if 'deployment_file' not in locals(): deployment_file = "deployment.yaml"
    if 'service_file' not in locals(): service_file = "service.yaml"
    if 'APP_NAME' not in locals(): APP_NAME = "django-app"
    if 'SERVICE_NAME' not in locals(): SERVICE_NAME = "django-app-service"


# --- Logic (Instructions) ---
# Check if K8S_CLUSTER_NAME was retrieved or defaulted reasonably before printing instructions
if K8S_CLUSTER_NAME == "<K8S_CLUSTER_NAME_MISSING>":
     print("ERROR: Kubernetes Cluster Name not found. Cannot provide accurate instructions.")
     # Optionally sys.exit() here
else:
    print(f"--- ACTION REQUIRED (Local Machine - using kubectl configured for {K8S_CLUSTER_NAME}) ---")

    print(f"\n1. Ensure the generated '{deployment_file}' and '{service_file}' files are available locally.")
    print(f"   (Download from Colab's file browser if generated there, or copy/paste the YAML output from Cell 6).")

    print("\n2. Apply the Deployment:")
    print(f"   kubectl apply -f {deployment_file}")

    print("\n3. Apply the Service:")
    print(f"   kubectl apply -f {service_file}")

    print("\n4. Check the status:")
    print("   - Wait for pods to be created and running:")
    print(f"     kubectl get pods -l app={APP_NAME} -w")
    print("     (Press Ctrl+C to stop watching once pods are 'Running')")
    print("\n   - Check the deployment status:")
    print(f"     kubectl get deployment {APP_NAME}")
    print("\n   - **Get the Load Balancer IP:** Check the service and find the EXTERNAL-IP:")
    print(f"     kubectl get service {SERVICE_NAME}")
    print("     (It might take a minute or two for the EXTERNAL-IP to appear. Re-run if it shows '<pending>')")

    print("\n5. **CRITICAL:** Once the EXTERNAL-IP is assigned, copy it accurately. You MUST provide it in the next relevant cell (Cell 8 for DNS).")
    print("------------------------------------------------------------------------------------\n")

# Initialize LOAD_BALANCER_IP - user will input in the next cell if needed
LOAD_BALANCER_IP = ""
# Store the empty variable, comment is now on a separate line
%store LOAD_BALANCER_IP

print("Instructions displayed. Initialized LOAD_BALANCER_IP variable for the next step.")

In [ ]:
# @title Cell 7.5: Step 5 (Part 2 - OPTIONAL/ADVANCED) - Apply Manifests via Kubernetes API
# -----------------------------------------------------------------------------
# WARNING: AUTOMATING KUBECTL/API INTERACTIONS FROM COLAB IS COMPLEX AND
#          CARRIES SECURITY RISKS. IT'S GENERALLY RECOMMENDED TO USE MANUAL
#          `kubectl apply` LOCALLY OR DEDICATED CD TOOLS FOR PRODUCTION.
#          THIS CELL IS FOR DEMONSTRATION PURPOSES.
#
# Goal: Attempt to apply the generated manifests (Secret, Deployment, Service)
#       directly to the Kubernetes cluster using the Python client library.
# Requires:
#   - `kubernetes` Python library installed.
#   - `doctl` CLI installed (for fetching kubeconfig).
#   - DigitalOcean API Token accessible (e.g., from Cell 1).
#   - Generated manifest files (e.g., secret.yaml, deployment.yaml, service.yaml)
#     existing in the Colab environment.
#   - Stored variables: K8S_CLUSTER_NAME, SECRET_NAME, APP_NAME, SERVICE_NAME etc.
# -----------------------------------------------------------------------------

import os
import sys
import yaml
import subprocess
import time
# Import necessary kubernetes modules later, after installation checks

# --- Configuration ---
# Set to True to attempt execution, False to skip.
ENABLE_API_AUTOMATION = True # <--- CHANGE TO True TO TRY THIS CELL

# --- Check if execution is enabled ---
if not ENABLE_API_AUTOMATION:
    print("Skipping Optional Cell 7.5 because ENABLE_API_AUTOMATION is False.")
else:
    print("--- Attempting Kubernetes API Automation (Experimental) ---")

    # --- Retrieve necessary variables ---
    # Assume variables are loaded from %store or defined earlier
    # Example placeholder assignments (replace with actual loading logic like %store -r)
    K8S_CLUSTER_NAME = None
    DIGITALOCEAN_TOKEN = None
    secret_file = "secret.yaml"
    deployment_file = "deployment.yaml"
    service_file = "service.yaml"
    SECRET_NAME = "my-app-secret" # Example
    APP_NAME = "my-app"       # Example
    SERVICE_NAME = "my-app-service" # Example

    # --- !! IMPORTANT !! ---
    # In a real Colab environment, uncomment the %store lines or ensure these
    # variables are correctly populated from previous cells.
    print("Attempting to load variables from storage...")
    try:
        # %store -r K8S_CLUSTER_NAME
        # %store -r DIGITALOCEAN_TOKEN
        # %store -r secret_file
        # %store -r SECRET_NAME
        # %store -r deployment_file
        # %store -r APP_NAME
        # %store -r service_file
        # %store -r SERVICE_NAME

        # Check required variables after attempting to load them
        if not K8S_CLUSTER_NAME or K8S_CLUSTER_NAME == "<K8S_CLUSTER_NAME_MISSING>": # Check for placeholder too
             raise ValueError("K8S_CLUSTER_NAME is missing or not properly set.")
        if not DIGITALOCEAN_TOKEN:
             raise ValueError("DIGITALOCEAN_TOKEN is missing (needed for doctl auth).")
        if not secret_file or not os.path.exists(secret_file):
            raise FileNotFoundError(f"Required manifest file not found: {secret_file}")
        if not deployment_file or not os.path.exists(deployment_file):
            raise FileNotFoundError(f"Required manifest file not found: {deployment_file}")
        if not service_file or not os.path.exists(service_file):
            raise FileNotFoundError(f"Required manifest file not found: {service_file}")
        if not SECRET_NAME or not APP_NAME or not SERVICE_NAME:
             raise ValueError("One or more resource names (SECRET_NAME, APP_NAME, SERVICE_NAME) are missing.")

        print("Successfully loaded required variables and found manifest files.")

    except (NameError, KeyError, FileNotFoundError, ValueError) as e:
        print(f"\nERROR: Missing required variables or files from previous steps: {e}")
        print("Ensure variables like K8S_CLUSTER_NAME, DIGITALOCEAN_TOKEN, etc., are set")
        print("and manifest files (secret.yaml, deployment.yaml, service.yaml) exist.")
        print("Cannot proceed with automation.")
        # Exit or prevent further execution
        ENABLE_API_AUTOMATION = False # Prevent further steps in this block
        # Alternatively, use sys.exit:
        # sys.exit("Stopping automation attempt due to missing prerequisites.")

# --- Continue only if automation is enabled and prerequisites are met ---
if ENABLE_API_AUTOMATION:

    # --- 1. Install Tools ---
    print("\n--- 1. Installing Tools ---")
    print("Installing Python kubernetes client...")
    # Use subprocess for pip install to avoid potential ! issues in some environments
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "kubernetes", "-q"], check=True)
        print("Python kubernetes client installed successfully.")
    except subprocess.CalledProcessError as e:
        print(f"ERROR installing kubernetes client: {e}")
        sys.exit("Kubernetes client installation failed.")
    except Exception as e:
        print(f"Unexpected error installing kubernetes client: {e}")
        sys.exit("Kubernetes client installation failed.")


    print("Checking/Installing doctl...")
    doctl_path = '/usr/local/bin/doctl'
    if os.path.exists(doctl_path):
         print("doctl already appears to be installed.")
         # Optional: You could add a version check here if needed
         # try:
         #   result = subprocess.run([doctl_path, "version"], capture_output=True, text=True, check=True)
         #   print(f"Existing doctl version info:\n{result.stdout}")
         # except Exception as e:
         #   print(f"Could not run 'doctl version': {e}")
    else:
         # Attempt robust doctl installation
         print("Attempting doctl installation...")
         doctl_archive = "doctl-linux-amd64.tar.gz"
         # Use specific version URL or latest. Latest can sometimes be unstable if format changes.
         # Using specific version is generally safer for automation.
         # Find latest version at: https://github.com/digitalocean/doctl/releases/latest
         # Example using latest (original approach):
         doctl_url = f"https://github.com/digitalocean/doctl/releases/latest/download/{doctl_archive}"
         # Example using a specific version (replace X.Y.Z with a real version like 1.104.0):
         # doctl_version = "1.104.0"
         # doctl_archive = f"doctl-{doctl_version}-linux-amd64.tar.gz"
         # doctl_url = f"https://github.com/digitalocean/doctl/releases/download/v{doctl_version}/{doctl_archive}"


         try:
             # 1. Download using curl, saving to a file
             print(f"Downloading doctl from {doctl_url}...")
             # Use -f to fail silently on server errors (important for check=True)
             curl_cmd = ["curl", "-fsSL", "-o", doctl_archive, doctl_url]
             # Add timeout to prevent hangs
             result = subprocess.run(curl_cmd, check=True, capture_output=True, text=True, timeout=300) # 5 min timeout
             print("Download complete.")

             # 2. Extract the archive
             print(f"Extracting {doctl_archive}...")
             # Extract directly into the current directory, specifically the 'doctl' binary
             tar_cmd = ["tar", "-xzvf", doctl_archive, "doctl"]
             result = subprocess.run(tar_cmd, check=True, capture_output=True, text=True)
             print("Extraction complete.")
             # print(result.stdout) # Optional: show tar output

             # 3. Move the binary to /usr/local/bin (requires permissions)
             print(f"Moving doctl to {doctl_path}...")
             # Using sudo might be necessary depending on Colab/environment permissions.
             # If sudo isn't available or needed, adjust the path (e.g., /content/bin)
             # and ensure that path is in the system's $PATH environment variable.
             # Trying without sudo first.
             mv_cmd = ["mv", "doctl", doctl_path]
             result = subprocess.run(mv_cmd, check=True, capture_output=True, text=True)
             print("Move complete.")

             # 4. Make executable (good practice)
             chmod_cmd = ["chmod", "+x", doctl_path]
             subprocess.run(chmod_cmd, check=True)
             print(f"{doctl_path} is now executable.")

             # 5. Clean up the downloaded archive
             print(f"Cleaning up {doctl_archive}...")
             os.remove(doctl_archive)
             print("Cleanup complete.")

             print("doctl installation successful.")

         except subprocess.TimeoutExpired:
             print(f"ERROR: Timeout occurred during doctl download/installation step.")
             if os.path.exists(doctl_archive): os.remove(doctl_archive)
             if os.path.exists("./doctl"): os.remove("./doctl")
             sys.exit("doctl installation failed due to timeout.")
         except subprocess.CalledProcessError as e:
             print(f"ERROR during doctl installation step: {' '.join(e.cmd)}")
             print(f"Exit Code: {e.returncode}")
             # Try to decode stderr/stdout for better error messages
             stderr_output = e.stderr.decode('utf-8', errors='ignore') if isinstance(e.stderr, bytes) else e.stderr
             stdout_output = e.stdout.decode('utf-8', errors='ignore') if isinstance(e.stdout, bytes) else e.stdout
             print(f"Stderr:\n{stderr_output}")
             print(f"Stdout:\n{stdout_output}")
             # Clean up partial downloads/extractions if they exist
             if os.path.exists(doctl_archive): os.remove(doctl_archive)
             if os.path.exists("./doctl"): os.remove("./doctl") # If extracted but not moved
             sys.exit("doctl installation failed.")
         except Exception as e:
             print(f"Unexpected error during doctl installation: {type(e).__name__} - {e}")
             # Clean up partial downloads/extractions
             if os.path.exists(doctl_archive): os.remove(doctl_archive)
             if os.path.exists("./doctl"): os.remove("./doctl")
             sys.exit("doctl installation failed.")

    # Import necessary kubernetes modules *after* successful installation checks
    try:
        from kubernetes import client, config
        from kubernetes.client.rest import ApiException
        # Uncomment if using kubernetes.utils.create_from_yaml
        # import kubernetes.utils
        print("Kubernetes library imported successfully.")
    except ImportError as e:
         print(f"ERROR importing kubernetes library: {e}")
         sys.exit("Failed to import Kubernetes library after installation.")

    # --- 2. Authenticate doctl and Fetch Kubeconfig ---
    print("\n--- 2. Authenticating and Fetching Kubeconfig ---")
    kubeconfig_path = "/content/kubeconfig.yaml" # Store kubeconfig temporarily in Colab
    try:
        # Set DO token as env var for doctl commands that need auth
        # Ensure the token is actually set from the variable check above
        do_env = os.environ.copy()
        do_env['DIGITALOCEAN_ACCESS_TOKEN'] = DIGITALOCEAN_TOKEN

        # Optional: Authenticate doctl explicitly if needed (often token env var is enough)
        # print("Running doctl auth init...")
        # auth_cmd = ["doctl", "auth", "init", "--access-token", DIGITALOCEAN_TOKEN]
        # result = subprocess.run(auth_cmd, check=True, capture_output=True, text=True, env=do_env)
        # print("doctl auth init successful.")

        # Fetch kubeconfig using 'show' and write manually (safer than shell redirection)
        print(f"Fetching kubeconfig for cluster '{K8S_CLUSTER_NAME}'...")
        cmd_show = ["doctl", "kubernetes", "cluster", "kubeconfig", "show", K8S_CLUSTER_NAME]
        result = subprocess.run(cmd_show, check=True, capture_output=True, text=True, env=do_env) # Pass env

        print("Kubeconfig content fetched successfully.")

        # Write the captured output to the file
        with open(kubeconfig_path, "w") as f:
            f.write(result.stdout)
        print(f"Kubeconfig saved to {kubeconfig_path}")

        # Add a small delay or check for file existence
        time.sleep(1) # Short delay might still help ensure filesystem sync
        if not os.path.exists(kubeconfig_path) or os.path.getsize(kubeconfig_path) == 0:
             raise FileNotFoundError(f"Kubeconfig file {kubeconfig_path} not found or empty after doctl command.")

    except subprocess.CalledProcessError as e:
        cmd_str = ' '.join(e.cmd) if isinstance(e.cmd, list) else e.cmd
        print(f"ERROR running doctl command: {cmd_str}")
        stderr_output = e.stderr.decode('utf-8', errors='ignore') if isinstance(e.stderr, bytes) else e.stderr
        stdout_output = e.stdout.decode('utf-8', errors='ignore') if isinstance(e.stdout, bytes) else e.stdout
        print(f"Stderr:\n{stderr_output}")
        print(f"Stdout:\n{stdout_output}")
        print("Ensure DIGITALOCEAN_TOKEN is valid, has permissions, and cluster name is correct.")
        sys.exit("Kubeconfig retrieval failed.")
    except FileNotFoundError as e:
         print(f"ERROR: {e}")
         sys.exit("Kubeconfig retrieval failed (file not found post-creation check).")
    except Exception as e:
        print(f"Unexpected ERROR during authentication/kubeconfig fetch: {type(e).__name__} - {e}")
        sys.exit("Kubeconfig retrieval failed.")

    # --- 3. Load Kubeconfig and Initialize Kubernetes API Client ---
    print("\n--- 3. Loading Kubeconfig and Initializing API Client ---")
    try:
        print(f"Loading kubeconfig from: {kubeconfig_path}")
        config.load_kube_config(config_file=kubeconfig_path)
        print("Kubernetes client configured.")
        # Initialize API clients needed
        core_v1_api = client.CoreV1Api()
        apps_v1_api = client.AppsV1Api()
        # networking_v1_api = client.NetworkingV1Api() # Needed for Ingress later
        print("API clients initialized (CoreV1Api, AppsV1Api).")
    except Exception as e:
        print(f"ERROR loading kubeconfig or initializing Kubernetes client: {e}")
        # Clean up sensitive file if loading failed
        if os.path.exists(kubeconfig_path):
            print(f"Cleaning up potentially sensitive kubeconfig file: {kubeconfig_path}")
            os.remove(kubeconfig_path)
        sys.exit("Kubernetes client initialization failed.")

    # --- 4. Apply Manifests using Python Client ---
    print("\n--- 4. Applying Manifests ---")
    namespace = "default" # Assuming default namespace, adjust if needed

    # Function to apply a single YAML file using Create/Patch logic
    def apply_manifest_file(file_path, core_api: client.CoreV1Api, apps_api: client.AppsV1Api):
        print(f"Applying manifest: {file_path}...")
        manifest_dict = None
        try:
            with open(file_path, 'r') as f:
                # Use safe_load_all for potentially multi-document YAMLs, though we expect one
                docs = list(yaml.safe_load_all(f))
                if not docs or docs[0] is None: # Check for empty file or just comments
                    print(f"  WARNING: Manifest file {file_path} is empty or contains no valid YAML document. Skipping.")
                    return True # Treat as success (nothing to apply)
                if len(docs) > 1:
                    print(f"  WARNING: Manifest file {file_path} contains multiple documents. Only applying the first one.")
                manifest_dict = docs[0]

            kind = manifest_dict.get('kind')
            api_version = manifest_dict.get('apiVersion')
            metadata = manifest_dict.get('metadata', {})
            metadata_name = metadata.get('name')
            resource_namespace = metadata.get('namespace', namespace) # Use manifest namespace or default

            if not kind or not metadata_name or not api_version:
                 raise ValueError(f"Could not parse kind, apiVersion, or metadata.name from {file_path}")

            print(f"  Kind: {kind}, Name: {metadata_name}, Namespace: {resource_namespace}, APIVersion: {api_version}")

            create_func = None
            patch_func = None
            read_func = None
            api_instance = None

            # Map Kind/APIVersion to API methods
            if kind == "Secret" and api_version == "v1":
                api_instance = core_api
                create_func = api_instance.create_namespaced_secret
                patch_func = api_instance.patch_namespaced_secret # Simple patch, might need strategic merge for complex updates
                read_func = api_instance.read_namespaced_secret
            elif kind == "Deployment" and api_version == "apps/v1":
                 api_instance = apps_api
                 create_func = api_instance.create_namespaced_deployment
                 patch_func = api_instance.patch_namespaced_deployment # Simple patch
                 read_func = api_instance.read_namespaced_deployment
            elif kind == "Service" and api_version == "v1":
                 api_instance = core_api
                 create_func = api_instance.create_namespaced_service
                 patch_func = api_instance.patch_namespaced_service # Simple patch
                 read_func = api_instance.read_namespaced_service
            # Add elif for other kinds like Ingress, ConfigMap, etc. as needed
            # elif kind == "Ingress" and api_version == "networking.k8s.io/v1":
            #      api_instance = client.NetworkingV1Api() # Ensure this was initialized if needed
            #      create_func = api_instance.create_namespaced_ingress
            #      patch_func = api_instance.patch_namespaced_ingress
            #      read_func = api_instance.read_namespaced_ingress
            else:
                print(f"  WARNING: Apply logic not implemented for Kind '{kind}/{api_version}'. Skipping API apply for {file_path}.")
                # If using kubernetes.utils.create_from_yaml, it might handle this kind.
                # Uncomment below and comment out manual logic if preferred (needs api_client general instance):
                # try:
                #    print(f"  Attempting apply via kubernetes.utils.create_from_yaml for {kind}/{api_version}")
                #    k8s_objects = kubernetes.utils.create_from_yaml(api_instance.api_client, file_path, verbose=True, namespace=resource_namespace)
                #    print(f"  Successfully applied objects from {file_path} via utils.")
                #    return True
                # except Exception as util_e:
                #    print(f"  ERROR using kubernetes.utils.create_from_yaml for {file_path}: {util_e}")
                #    return False
                return True # Treat unknown types as "skipped successfully" in this context

            # Proceed with Create or Patch logic
            if api_instance and create_func and patch_func and read_func:
                try:
                    # Check if it exists
                    print(f"  Checking if {kind} '{metadata_name}' exists in namespace '{resource_namespace}'...")
                    read_func(metadata_name, resource_namespace)
                    # Exists, so patch it
                    print(f"  Resource '{metadata_name}' exists. Patching...")
                    # Using simple patch. For more complex updates, consider strategic merge patch
                    # or server-side apply (_content_type='application/apply-patch+yaml')
                    patch_func(metadata_name, resource_namespace, manifest_dict)
                    print(f"  Patched {kind} '{metadata_name}'.")

                except ApiException as e:
                    if e.status == 404:
                        # Doesn't exist, create it
                        print(f"  Resource '{metadata_name}' not found. Creating...")
                        create_func(resource_namespace, manifest_dict)
                        print(f"  Created {kind} '{metadata_name}'.")
                    else:
                        # Other API error during read or patch
                        print(f"  ERROR checking/patching {kind} '{metadata_name}': Status {e.status}, Reason: {e.reason}")
                        try:
                            body = yaml.safe_load(e.body) # Try loading error body as yaml/json
                            print(f"  Error Body: {body}")
                        except:
                            print(f"  Error Body: {e.body}")
                        return False # Failed
            return True # Succeeded for known types

        except ApiException as e:
             print(f"ERROR applying manifest from {file_path}. Status: {e.status}, Reason: {e.reason}")
             try:
                 body = yaml.safe_load(e.body) # Try loading error body as yaml/json
                 print(f"Error Body: {body}")
             except:
                 print(f"Error Body: {e.body}")
             return False # Failed
        except FileNotFoundError:
             print(f"ERROR: Manifest file not found at {file_path}")
             return False
        except yaml.YAMLError as e:
             print(f"ERROR parsing YAML file {file_path}: {e}")
             return False
        except ValueError as e: # Catch parsing errors from our code
             print(f"ERROR processing manifest {file_path}: {e}")
             return False
        except Exception as e:
            # Catch potential issues like network errors during API calls, etc.
            print(f"Unexpected ERROR applying manifest {file_path}: {type(e).__name__} - {e}")
            # Include traceback for unexpected errors if needed for debugging
            # import traceback
            # traceback.print_exc()
            return False # Failed


    # Apply the manifests in a sensible order (Secrets often first)
    apply_success_flags = []
    apply_success_flags.append(apply_manifest_file(secret_file, core_v1_api, apps_v1_api))
    apply_success_flags.append(apply_manifest_file(deployment_file, core_v1_api, apps_v1_api))
    apply_success_flags.append(apply_manifest_file(service_file, core_v1_api, apps_v1_api))
    # Add apply for ingress_file later if needed
    # apply_success_flags.append(apply_manifest_file(ingress_file, core_v1_api, apps_v1_api)) # Requires NetworkingV1Api

    # Overall success is true only if all apply calls returned True
    overall_apply_success = all(apply_success_flags)

    # --- 5. Cleanup and Status ---
    print("\n--- 5. Cleanup and Final Status ---")
    # Clean up the temporary kubeconfig file (important for security)
    if os.path.exists(kubeconfig_path):
        try:
            print(f"Removing temporary kubeconfig: {kubeconfig_path}")
            os.remove(kubeconfig_path)
        except Exception as e:
            print(f"Warning: Could not remove temporary kubeconfig file {kubeconfig_path}: {e}")

    # Unset the environment variable if it was set for subprocesses
    # Note: os.environ changes might not persist reliably across Colab cells anyway,
    # but it's good practice if the script were run standalone.
    if 'DIGITALOCEAN_ACCESS_TOKEN' in os.environ:
         # This might not be strictly necessary if using the 'env' param in subprocess
         # but doesn't hurt.
         # print("Note: Unsetting DIGITALOCEAN_ACCESS_TOKEN from os.environ (may not be effective across cells).")
         # del os.environ['DIGITALOCEAN_ACCESS_TOKEN']
         pass # Avoid deleting as it might be needed by subsequent cells if not using %store

    if overall_apply_success:
        print("\nAll specified manifests applied via API (or skipped unimplemented types).")
        print("Check the status of your resources using manual kubectl commands or the DigitalOcean UI.")
        print("Example checks (run these manually or in a separate cell with kubectl configured):")
        print(f"  kubectl get secret {SECRET_NAME} -n {namespace}")
        print(f"  kubectl get deployment {APP_NAME} -n {namespace}")
        print(f"  kubectl get pods -l app={APP_NAME} -n {namespace}")
        print(f"  kubectl get service {SERVICE_NAME} -n {namespace}")
    else:
        print("\nOne or more manifests failed to apply via the API. Review the errors above.")
        print("You may need to check your cluster state and apply the remaining manifests manually.")
        print("Use `kubectl apply -f <filename>.yaml` for manual application.")

# Final message regardless of whether automation was enabled or prerequisites were met
print("\nOptional Cell 7.5 finished.")

In [ ]:
# @title Cell 8: Step 6 (Part 1) - Configure DNS (Manual or Automated)
"""
This cell helps configure a DNS 'A' record to point a domain or subdomain
to the IP address of a previously configured Load Balancer.

It supports two modes:
1. Automated (via DigitalOcean API): If your domain's DNS is managed by
   DigitalOcean (`IS_DOMAIN_MANAGED_BY_DO = True`).
2. Manual Instructions: If your domain's DNS is managed elsewhere
   (`IS_DOMAIN_MANAGED_BY_DO = False`).

This step is optional but necessary if you want to access your application
via a custom domain/subdomain.

Prerequisites:
- A domain name (registered anywhere).
- A DigitalOcean Load Balancer should have been created, and its external IP known.
- (For Automated Mode):
    - A DigitalOcean account with API access configured.
    - The domain name MUST be managed through DigitalOcean DNS.
    - The following variables must have been stored from previous cells using %store:
        - DO_API_BASE_URL
        - HEADERS
        - check_do_response helper function
- The Load Balancer's external IP address (the script will prompt for this).

Configuration:
- Set `CONFIGURE_DOMAIN` to True to execute this cell's logic.
- Set `IS_DOMAIN_MANAGED_BY_DO` to True if DO manages your DNS, False otherwise.
- Replace `"your-domain.com"` in `DOMAIN_NAME` with your actual domain.
- Set `DNS_RECORD_NAME` to the desired record name:
    - "@" points the root domain (e.g., your-domain.com).
    - "www" points a subdomain (e.g., www.your-domain.com).
    - Any other string creates a different subdomain (e.g., "app" for app.your-domain.com).

What it does:
1. Checks if DNS configuration is enabled (`CONFIGURE_DOMAIN`).
2. Prompts the user to enter the Load Balancer's external IP address.
3. **Conditional Logic:**
    - **If `IS_DOMAIN_MANAGED_BY_DO` is True:**
        - Constructs and sends a request to the DigitalOcean API to create the DNS 'A' record.
        - Handles potential API errors (e.g., record already exists).
    - **If `IS_DOMAIN_MANAGED_BY_DO` is False:**
        - Skips the API call.
        - Prints detailed instructions for the user to manually create the 'A' record
          at their actual DNS provider (e.g., GoDaddy, Cloudflare, Namecheap).
4. Stores `DOMAIN_NAME`, `CONFIGURE_DOMAIN`, and the confirmed `LOAD_BALANCER_IP`
   for potential use in subsequent steps (like HTTPS setup).
"""

# --- Imports ---
import requests  # For making HTTP requests to the DO API (if applicable)
import sys       # For exiting the script cleanly on critical errors

# --- Retrieve stored variables needed ---
# %store -r retrieves variables saved in previous notebook cells.
# HEADERS, DO_API_BASE_URL, check_do_response are only needed if IS_DOMAIN_MANAGED_BY_DO is True
# We'll retrieve them conditionally or handle potential NameErrors later if needed.
# LOAD_BALANCER_IP might exist, but we'll prioritize user input.
try:
    %store -r DO_API_BASE_URL HEADERS check_do_response LOAD_BALANCER_IP
except (NameError, KeyError):
    # Handle cases where these might not have been stored if prior steps were skipped
    # Assign default/None values if they are crucial *before* the conditional check
    # Or rely on the conditional check to prevent their use if IS_DOMAIN_MANAGED_BY_DO is False
    print("Note: Some stored variables (e.g., DO_API_BASE_URL, HEADERS) might not be loaded,")
    print("      which is expected if prior DigitalOcean-specific steps were skipped.")
    if 'LOAD_BALANCER_IP' not in locals(): # Ensure LOAD_BALANCER_IP exists even if not stored
        LOAD_BALANCER_IP = None

# --- Configuration for this Step ---

# Set to True to run the DNS configuration logic/instructions, False to skip.
CONFIGURE_DOMAIN = False # <--- SET TO True or False

# *** NEW FLAG ***
# Set to True ONLY if your domain (DOMAIN_NAME) uses DigitalOcean's nameservers.
# Set to False if your DNS is managed elsewhere (GoDaddy, Cloudflare, Namecheap, etc.).
IS_DOMAIN_MANAGED_BY_DO = False # <--- SET TO True or False

# IMPORTANT: Replace with the domain name you want to configure.
DOMAIN_NAME = "your-domain.com" # <--- REPLACE with your domain name

# The name of the DNS record.
# Use "@" for the root domain (e.g., your-domain.com).
# Use a string like "www" or "app" for a subdomain (e.g., www.your-domain.com).
DNS_RECORD_NAME = "@" # <--- Set to "@" or your desired subdomain (e.g., "www", "app")

# --- Main Logic ---
if not CONFIGURE_DOMAIN:
    print("Skipping DNS configuration because CONFIGURE_DOMAIN is set to False.")
else:
    print(f"Starting DNS configuration process for: {DNS_RECORD_NAME}.{DOMAIN_NAME}")

    # Validate that a domain name has been provided
    if not DOMAIN_NAME or DOMAIN_NAME == "your-domain.com":
        print("\nERROR: DOMAIN_NAME is not set or is still the placeholder value.")
        print("       Please update the DOMAIN_NAME variable in this cell.")
        sys.exit("DNS configuration failed due to missing domain name.") # Exit if no valid domain

    # --- Get Load Balancer IP from User ---
    print(f"\nThis step requires the EXTERNAL IP address of your Load Balancer.")
    if IS_DOMAIN_MANAGED_BY_DO:
        print(f"If using DigitalOcean DNS, the 'A' record for '{DNS_RECORD_NAME}.{DOMAIN_NAME}' will be pointed to this IP via the API.")
    else:
        print(f"You will need to manually create a DNS 'A' record at your provider for")
        print(f"'{DNS_RECORD_NAME}.{DOMAIN_NAME}' pointing to this IP.")

    # Always ask for IP to ensure it's current, regardless of management method
    user_input_ip = input(f"Please paste the Load Balancer's External IP Address: ").strip()

    # Validate user input
    if not user_input_ip:
        print("\nERROR: Load Balancer IP address was not provided.")
        sys.exit("DNS configuration failed due to missing IP address.") # Exit if no IP given

    # Use the IP provided by the user
    LOAD_BALANCER_IP = user_input_ip
    # Store the confirmed IP address for potential use later
    %store LOAD_BALANCER_IP
    print(f"Using IP Address: {LOAD_BALANCER_IP} for DNS configuration.")

    # --- Conditional DNS Configuration ---
    if IS_DOMAIN_MANAGED_BY_DO:
        print(f"\nAttempting automated DNS configuration via DigitalOcean API...")

        # Check if necessary DO variables were loaded
        if 'DO_API_BASE_URL' not in locals() or 'HEADERS' not in locals() or 'check_do_response' not in locals():
             print("\nERROR: Required DigitalOcean API variables (DO_API_BASE_URL, HEADERS, check_do_response)")
             print("       were not found. Ensure previous cells that define these were run successfully.")
             sys.exit("Automated DNS configuration failed due to missing DO API variables.")

        # --- Create DNS 'A' Record via DigitalOcean API ---
        dns_endpoint = f"{DO_API_BASE_URL}/domains/{DOMAIN_NAME}/records"
        payload = {
            "type": "A",
            "name": DNS_RECORD_NAME,
            "data": LOAD_BALANCER_IP,
            "ttl": 1800
        }

        print(f"Creating DNS 'A' record: {payload['name']}.{DOMAIN_NAME} -> {payload['data']} (TTL: {payload['ttl']}s)")

        try:
            response = requests.post(dns_endpoint, headers=HEADERS, json=payload)

            # Handle API Response (including 422 for existing record)
            if response.status_code == 404:
                 print(f"WARNING: Received HTTP 404 (Not Found) from DigitalOcean API.")
                 print(f"         This likely means the domain '{DOMAIN_NAME}' is NOT managed by DigitalOcean DNS,")
                 print(f"         or there's a typo in the DOMAIN_NAME variable.")
                 print(f"         Please verify DOMAIN_NAME and ensure IS_DOMAIN_MANAGED_BY_DO is set correctly.")
                 print(f"         If the domain isn't managed by DO, set IS_DOMAIN_MANAGED_BY_DO = False and re-run.")
                 # Consider providing manual instructions here as well, or exiting
                 print("\n--- Switching to Manual DNS Instructions ---")
                 IS_DOMAIN_MANAGED_BY_DO = False # Flip the flag to show manual instructions below

            elif response.status_code == 422:
                print(f"WARNING: Received HTTP 422 from DigitalOcean API.")
                print(f"         This usually means a DNS record for '{payload['name']}.{DOMAIN_NAME}' of type 'A' already exists.")
                print(f"         If the existing record points to the correct IP ({payload['data']}), no action is needed.")
                print(f"         If it points to the WRONG IP, you may need to manually update or delete")
                print(f"         the old record in the DigitalOcean control panel (Networking -> Domains -> {DOMAIN_NAME})")
                print(f"         and optionally re-run this cell.")
            else:
                # Use the helper function for other statuses
                result = check_do_response(response, f"Create DNS Record for {DOMAIN_NAME}")
                if result and 'domain_record' in result:
                    record_id = result['domain_record'].get('id', 'N/A')
                    print(f"DNS 'A' record created/processed successfully via DO API. (Record ID: {record_id})")
                else:
                    print(f"Automated DNS record creation failed or status could not be confirmed.")
                    # check_do_response should have printed error details.

        except requests.exceptions.RequestException as e:
            print(f"\nNETWORK ERROR during API call: {e}")
            print("  Please check internet connection and DO API status.")
        except Exception as e:
             print(f"\nUNEXPECTED ERROR during API call: {e}")

    # --- Manual Configuration Instructions ---
    # This block executes if IS_DOMAIN_MANAGED_BY_DO was initially False,
    # OR if the API call resulted in a 404 (indicating domain not found at DO).
    if not IS_DOMAIN_MANAGED_BY_DO:
        print("\n" + "="*40)
        print("--- MANUAL DNS Configuration Required ---")
        print("="*40)
        print(f"\nBecause your domain '{DOMAIN_NAME}' is not managed by DigitalOcean's DNS,")
        print("you need to manually create a DNS record with your DNS provider.")
        print("(Examples: GoDaddy, Cloudflare, Namecheap, Google Domains, etc.)")

        # Determine the full hostname/subdomain to configure
        record_display_name = DOMAIN_NAME if DNS_RECORD_NAME == "@" else f"{DNS_RECORD_NAME}.{DOMAIN_NAME}"
        # Determine how the 'Name' or 'Host' field should likely be entered at the provider
        # Common variations: "@" or empty for root, "subdomain" for sub.domain.com
        host_name_for_provider = DNS_RECORD_NAME # Most providers accept "@" or the subdomain name directly

        print("\nLog in to your DNS provider's control panel and add the following record:")
        print(f"  - Record Type:       A")
        print(f"  - Name / Host / Alias: {host_name_for_provider}")
        print(f"     (Use '@' or leave blank if setting for the root domain '{DOMAIN_NAME}',")
        print(f"      or use '{DNS_RECORD_NAME}' if setting for the subdomain '{record_display_name}'.")
        print(f"      Check your provider's documentation if unsure.)")
        print(f"  - Value / Points to: {LOAD_BALANCER_IP}")
        print(f"  - TTL (Time-To-Live): Use your provider's default, or set to 1800 (30 minutes)")
        print(f"                         or 3600 (1 hour). Lower TTLs propagate faster.")

        print(f"\nSummary: You are pointing '{record_display_name}' to the IP address '{LOAD_BALANCER_IP}'.")

    # --- Final Notes ---
    print("\nIMPORTANT: DNS changes (whether made by API or manually) can take time")
    print("           (minutes to several hours) to propagate across the internet.")
    print("           You might not be able to access your domain/subdomain via the")
    print(f"           new IP ({LOAD_BALANCER_IP}) immediately.")
    print("           You can use online tools like 'DNS Checker' to monitor propagation.")

# Store configuration used/confirmed in this step for potential use later.
# Storing LOAD_BALANCER_IP is already done above after user input.
%store DOMAIN_NAME CONFIGURE_DOMAIN

print("\nStep 6 (Part 1) - DNS Configuration guidance finished.")

In [ ]:
# @title Cell 9: Step 6 (Part 2) - Setup Cert-Manager & Ingress (Instructions & Generation)

# --- Imports ---
import yaml
# --- Retrieve stored variables needed ---
%store -r DOMAIN_NAME APP_NAME SERVICE_NAME K8S_CLUSTER_NAME CONFIGURE_DOMAIN

# --- Configuration for this Step ---
# Ensure CONFIGURE_DOMAIN and DOMAIN_NAME are set if you want Ingress/TLS
CLUSTER_ISSUER_NAME = "letsencrypt-prod" # <--- Name for your ClusterIssuer (e.g., letsencrypt-prod, letsencrypt-staging)
INGRESS_CLASS_NAME = "nginx" # <--- Set your Ingress Controller class (e.g., nginx, traefik)
YOUR_EMAIL_ADDRESS = "anand.butani@gmail.com" # <--- REPLACE with your email for Let's Encrypt

# --- Logic ---
if not CONFIGURE_DOMAIN or not DOMAIN_NAME:
    print("Skipping Cert-Manager/Ingress setup because CONFIGURE_DOMAIN is False or DOMAIN_NAME is not set.")
else:
    print(f"--- ACTION REQUIRED (Local Machine - using kubectl configured for {K8S_CLUSTER_NAME}) ---")
    print("\nPart A: Install Cert-Manager (One-time setup per cluster)")
    print("1. If Cert-Manager is NOT already installed:")
    print("   Follow the official instructions: https://cert-manager.io/docs/installation/")
    print("   Example (check for latest version!):")
    print("   kubectl apply -f https://github.com/cert-manager/cert-manager/releases/download/v1.14.5/cert-manager.yaml")
    print("   Wait for Cert-Manager pods to be ready:")
    print("   kubectl get pods -n cert-manager -w")
    print("2. If Cert-Manager IS installed, proceed to Part B.")

    print("\nPart B: Create a ClusterIssuer (One-time setup per issuer type)")
    print("1. If you DON'T have a suitable ClusterIssuer:")
    cluster_issuer_yaml = f"""
apiVersion: cert-manager.io/v1
kind: ClusterIssuer
metadata:
  name: {CLUSTER_ISSUER_NAME}
spec:
  acme:
    # Use EITHER the staging or production server:
    server: https://acme-v02.api.letsencrypt.org/directory # Production
    # server: https://acme-staging-v02.api.letsencrypt.org/directory # Staging (for testing)
    email: {YOUR_EMAIL_ADDRESS} # *** YOUR EMAIL HERE ***
    privateKeySecretRef:
      name: {CLUSTER_ISSUER_NAME}-private-key # Stores ACME account key
    solvers:
    # Choose ONE solver type based on your setup:
    # Option 1: HTTP01 (Requires an Ingress Controller like Nginx or Traefik)
    - http01:
        ingress:
          class: {INGRESS_CLASS_NAME} # Match your ingress class

    # Option 2: DNS01 (Requires configuring DNS provider credentials)
    # - dns01:
    #     digitalocean:
    #       tokenSecretRef:
    #         name: digitalocean-dns # Secret containing DO token with DNS write access
    #         key: access-token
"""
    print("\n   Define a ClusterIssuer like the example below (save as cluster-issuer.yaml):")
    print(cluster_issuer_yaml)
    print(f"   Apply it: kubectl apply -f cluster-issuer.yaml")
    print(f"   Verify: kubectl describe clusterissuer {CLUSTER_ISSUER_NAME}")
    print("2. If you HAVE a suitable ClusterIssuer, proceed to Part C.")


    print("\nPart C: Generate and Apply the Ingress Resource")
    # --- Define Ingress YAML ---
    ingress_manifest = {
        "apiVersion": "networking.k8s.io/v1",
        "kind": "Ingress",
        "metadata": {
            "name": f"{APP_NAME}-ingress",
            "annotations": {
                # Standard annotation for cert-manager
                "cert-manager.io/cluster-issuer": CLUSTER_ISSUER_NAME,
                # Specify ingress class if not default
                "kubernetes.io/ingress.class": INGRESS_CLASS_NAME,
                # Add other ingress controller specific annotations if needed
                # e.g., for Nginx force HTTPS redirect:
                # "nginx.ingress.kubernetes.io/force-ssl-redirect": "true",
            }
        },
        "spec": {
            "tls": [{
                "hosts": [DOMAIN_NAME],
                "secretName": f"{APP_NAME}-tls-secret" # Cert will be stored here
            }],
            "rules": [{
                "host": DOMAIN_NAME,
                "http": {
                    "paths": [{
                        "pathType": "Prefix",
                        "path": "/",
                        "backend": {
                            "service": {
                                "name": SERVICE_NAME, # Target the service from Cell 6
                                "port": {"number": 80} # Target service port
                            }
                        }
                    }]
                }
            }]
        }
    }

    ingress_file = "ingress.yaml"
    print(f"\n1. Generated Ingress Manifest ({ingress_file}):")
    print("---")
    print(yaml.dump(ingress_manifest))
    print("---")
    with open(ingress_file, 'w') as f:
        yaml.dump(ingress_manifest, f)
    print(f"\n2. Save the generated YAML above to '{ingress_file}' locally or download from Colab.")

    print(f"\n3. Apply the Ingress Manifest:")
    print(f"   kubectl apply -f {ingress_file}")

    print("\n4. Check Ingress and Certificate Status:")
    print("   - Check ingress:")
    print(f"     kubectl get ingress {APP_NAME}-ingress")
    print("   - Monitor certificate process (may take a few minutes):")
    print("     kubectl get certificate -w")
    print("     kubectl describe certificate <certificate-name>") # Get name from previous cmd
    print("     kubectl get events --sort-by='.lastTimestamp'") # Check for errors

    print(f"\n5. Once the certificate is issued and Ingress shows an ADDRESS, access your app:")
    print(f"   https://{DOMAIN_NAME}")
    print("-----------------------------------------------------------------------------------\n")

# Store filename for reference
%store ingress_file

#ToDo

## 1. Explore Database Operator
  ### 1a. Explore Cluster dedicated Redis

## 2. Setup Cluster Recognition
  ### Setup new Git/Docker deployments on existing clusters

## 3. Monitoring
  ### Dashboards
  ### Alerts

# TESTING API